# Virginia Regional Economic Intelligence

## 03 — Regional Economic Analysis

This notebook analyzes employment growth, industry structure, and regional specialization across Virginia using the quality-enriched QCEW panel developed in the previous notebooks.

### Objectives

- measure employment and wage change across Virginia localities,
- compare industry performance over time,
- identify sectors that are concentrated within particular local economies,
- distinguish statewide industry trends from locally specific performance,
- and develop interpretable regional indicators for policy-oriented analysis.

The analysis applies the data-quality rules established previously, including explicit treatment of disclosure-suppressed observations and minimum employment thresholds for percentage-growth rankings.

In [1]:
# Import core analytical libraries.

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Define project paths and load the quality-enriched analytical panel.
# -------------------------------------------------------------------
# Notebook 03 begins from the validated output of Notebook 02 rather
# than repeating the acquisition and quality-control workflow.

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

PANEL_PATH = (
    PROCESSED_DIR
    / "va_qcew_private_sector_panel_2019_2025_quality_enriched.csv"
)

panel = pd.read_csv(
    PANEL_PATH,
    dtype={
        "area_fips": str,
        "industry_code": str,
        "own_code": str,
        "agglvl_code": str,
        "disclosure_code": str
    }
)

print(f"Rows loaded: {len(panel):,}")
print(f"Study period: {panel['year'].min()}–{panel['year'].max()}")
print(f"Virginia localities: {panel['area_fips'].nunique()}")
print(f"NAICS sectors: {panel['industry_code'].nunique()}")

Rows loaded: 17,804
Study period: 2019–2025
Virginia localities: 133
NAICS sectors: 20


## 1. Statewide Industry Employment Trends

Regional performance should be interpreted relative to broader economic conditions.

Before evaluating individual localities, the analysis establishes the statewide industry context. This provides a benchmark for distinguishing between growth that reflects a broader Virginia-wide trend and growth that appears unusually strong or weak within a specific locality.

### Research Question

**How has private-sector employment changed across Virginia industries from 2019 to 2025?**

This question provides the statewide context needed for later regional comparisons. If a locality grows strongly in a particular industry, that growth may reflect either a broad expansion of the industry across Virginia or a more localized competitive advantage.

Establishing statewide industry trends first helps separate these two effects.

### Method

Published locality-level employment is aggregated by industry and year across Virginia.

Because disclosure-suppressed locality-industry observations contain unavailable employment values, the resulting totals represent the sum of **published locality observations** rather than official statewide QCEW employment totals.

These aggregates are therefore used as an internally consistent analytical benchmark for regional comparisons rather than as substitutes for official statewide estimates.

In [4]:
# Aggregate published employment across Virginia localities by industry and year.
# ------------------------------------------------------------------------------
# pandas sum() ignores suppressed observations because their employment values
# were converted to NaN during the data-quality workflow.
#
# These totals therefore represent published locality-level employment and are
# used to establish comparative industry trends within the analytical panel.

state_industry_employment = (
    panel
    .groupby(
        ["year", "industry_code", "industry_title"],
        as_index=False
    )
    .agg(
        published_employment=("annual_avg_emplvl", "sum"),
        published_localities=("annual_avg_emplvl", "count")
    )
)

state_industry_employment.head(20)

,year,industry_code,industry_title,published_employment,published_localities
0,2019,11,"Agriculture, Forestry, Fishing and Hunting",5163.0,50
1,2019,21,"Mining, Quarrying, and Oil and Gas Extraction",4121.0,19
2,2019,22,Utilities,4738.0,22
3,2019,23,Construction,185255.0,116
4,2019,31-33,Manufacturing,230859.0,123
5,2019,42,Wholesale Trade,76780.0,85
6,2019,44-45,Retail Trade,402123.0,132
7,2019,48-49,Transportation and Warehousing,112481.0,88
8,2019,51,Information,63064.0,100
9,2019,52,Finance and Insurance,119986.0,119


### 2019–2025 Statewide Industry Comparison

To evaluate structural change over the full study period, published employment in 2019 is compared with published employment in 2025 for each broad NAICS sector.

Both absolute employment change and percentage growth are calculated. Because these totals are constructed from published locality-level observations, changes may reflect both underlying economic activity and changes in disclosure coverage. The results are therefore interpreted alongside the number of localities with published employment in each year.

In [5]:
# Compare published statewide industry employment between 2019 and 2025.
# ----------------------------------------------------------------------
# The table is reshaped so that each industry has separate 2019 and 2025
# employment and locality-coverage values. This allows us to calculate
# long-term change while also monitoring whether publication coverage changed.

state_industry_comparison = (
    state_industry_employment[
        state_industry_employment["year"].isin([2019, 2025])
    ]
    .pivot(
        index=["industry_code", "industry_title"],
        columns="year",
        values=["published_employment", "published_localities"]
    )
)

# Flatten the multi-level column names created by the pivot.
state_industry_comparison.columns = [
    f"{measure}_{year}"
    for measure, year in state_industry_comparison.columns
]

state_industry_comparison = (
    state_industry_comparison
    .reset_index()
)

# Calculate absolute and percentage employment change.
state_industry_comparison["employment_change"] = (
    state_industry_comparison["published_employment_2025"]
    - state_industry_comparison["published_employment_2019"]
)

state_industry_comparison["employment_growth_pct"] = (
    state_industry_comparison["employment_change"]
    / state_industry_comparison["published_employment_2019"]
)

# Track how much locality-level publication coverage changed.
state_industry_comparison["published_locality_change"] = (
    state_industry_comparison["published_localities_2025"]
    - state_industry_comparison["published_localities_2019"]
)

state_industry_comparison.sort_values(
    "employment_growth_pct",
    ascending=False
)

,industry_code,industry_title,published_employment_2019,published_employment_2025,published_localities_2019,published_localities_2025,employment_change,employment_growth_pct,published_locality_change
2,22,Utilities,4738.0,6295.0,22.0,23.0,1557.0,0.328620,1.0
7,48-49,Transportation and Warehousing,112481.0,148300.0,88.0,80.0,35819.0,0.318445,-8.0
16,71,"Arts, Entertainment, and Recreation",56373.0,64859.0,105.0,94.0,8486.0,0.150533,-11.0
15,62,Health Care and Social Assistance,407114.0,459088.0,82.0,72.0,51974.0,0.127664,-10.0
12,55,Management of Companies and Enterprises,72822.0,80636.0,63.0,60.0,7814.0,0.107303,-3.0
10,53,Real Estate and Rental and Leasing,53573.0,58068.0,119.0,105.0,4495.0,0.083904,-14.0
14,61,Educational Services,57554.0,59467.0,69.0,69.0,1913.0,0.033238,0.0
3,23,Construction,185255.0,190104.0,116.0,87.0,4849.0,0.026175,-29.0
11,54,"Professional, Scientific, and Technical Services",393455.0,402656.0,99.0,95.0,9201.0,0.023385,-4.0
17,72,Accommodation and Food Services,342161.0,326759.0,109.0,98.0,-15402.0,-0.045014,-11.0


### Initial Interpretation

Published employment growth varies substantially across Virginia industries between 2019 and 2025.

Transportation and Warehousing shows particularly strong growth, increasing by approximately 31.8% despite employment being published for fewer localities in 2025 than in 2019. Health Care and Social Assistance also shows substantial published employment growth.

However, several apparent industry declines coincide with large reductions in locality-level publication coverage. Information, Agriculture, Manufacturing, and Unclassified employment are especially affected by this issue.

Because changes in disclosure coverage can influence aggregated employment totals, these raw comparisons should not be interpreted as definitive statewide industry growth rates. A matched-locality comparison is required to separate employment change from changes in publication coverage.

In [6]:
# Construct a matched-locality comparison for 2019 and 2025.
# -----------------------------------------------------------
# For each industry, only localities with published employment in BOTH
# endpoint years are retained. This prevents changes in disclosure
# coverage from being mistaken for actual employment growth.

matched_endpoint_panel = (
    panel[
        panel["year"].isin([2019, 2025])
        & panel["annual_avg_emplvl"].notna()
    ]
    [
        [
            "year",
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title",
            "annual_avg_emplvl"
        ]
    ]
)

# Count how many endpoint years are available for each
# locality × industry combination.
matched_series = (
    matched_endpoint_panel
    .groupby(
        ["area_fips", "industry_code"]
    )["year"]
    .nunique()
    .reset_index(name="endpoint_years_available")
)

# Keep only locality-industry combinations observed in both 2019 and 2025.
matched_series = matched_series[
    matched_series["endpoint_years_available"] == 2
]

matched_endpoint_panel = (
    matched_endpoint_panel
    .merge(
        matched_series[
            ["area_fips", "industry_code"]
        ],
        on=["area_fips", "industry_code"],
        how="inner",
        validate="many_to_one"
    )
)

print(
    f"Matched locality-industry endpoint observations: "
    f"{len(matched_endpoint_panel):,}"
)

print(
    f"Matched locality-industry series: "
    f"{len(matched_series):,}"
)

Matched locality-industry endpoint observations: 2,918
Matched locality-industry series: 1,459


### Matched-Locality Industry Growth

To reduce bias from changing disclosure coverage, statewide industry growth is recalculated using only locality-industry series with published employment in both 2019 and 2025.

This matched-sample approach holds the geographic composition constant across the comparison period. As a result, changes in aggregated employment are less likely to reflect shifts in publication coverage and more likely to represent underlying employment change within consistently observed local economies.

In [7]:
# Aggregate employment within the matched locality sample.
# --------------------------------------------------------
# Because each retained locality-industry series is observed in both
# 2019 and 2025, the comparison uses a consistent geographic sample.

matched_industry_employment = (
    matched_endpoint_panel
    .groupby(
        ["year", "industry_code", "industry_title"],
        as_index=False
    )
    .agg(
        matched_employment=("annual_avg_emplvl", "sum"),
        matched_localities=("area_fips", "nunique")
    )
)

# Reshape the endpoint totals so that each industry occupies one row.
matched_industry_comparison = (
    matched_industry_employment
    .pivot(
        index=["industry_code", "industry_title"],
        columns="year",
        values=["matched_employment", "matched_localities"]
    )
)

# Flatten the multi-level column names created by the pivot operation.
matched_industry_comparison.columns = [
    f"{measure}_{year}"
    for measure, year in matched_industry_comparison.columns
]

matched_industry_comparison = (
    matched_industry_comparison
    .reset_index()
)

# Calculate absolute and percentage employment change within the
# consistently observed locality sample.
matched_industry_comparison["employment_change"] = (
    matched_industry_comparison["matched_employment_2025"]
    - matched_industry_comparison["matched_employment_2019"]
)

matched_industry_comparison["employment_growth_pct"] = (
    matched_industry_comparison["employment_change"]
    / matched_industry_comparison["matched_employment_2019"]
)

matched_industry_comparison.sort_values(
    "employment_growth_pct",
    ascending=False
)

,industry_code,industry_title,matched_employment_2019,matched_employment_2025,matched_localities_2019,matched_localities_2025,employment_change,employment_growth_pct
7,48-49,Transportation and Warehousing,107633.0,143653.0,70.0,70.0,36020.0,0.334656
16,71,"Arts, Entertainment, and Recreation",53830.0,64790.0,91.0,91.0,10960.0,0.203604
19,99,Unclassified,5434.0,6349.0,49.0,49.0,915.0,0.168384
3,23,Construction,165261.0,187722.0,85.0,85.0,22461.0,0.135912
15,62,Health Care and Social Assistance,392163.0,444571.0,62.0,62.0,52408.0,0.133638
12,55,Management of Companies and Enterprises,72102.0,79546.0,54.0,54.0,7444.0,0.103243
10,53,Real Estate and Rental and Leasing,52744.0,57748.0,104.0,104.0,5004.0,0.094873
11,54,"Professional, Scientific, and Technical Services",390722.0,400870.0,84.0,84.0,10148.0,0.025972
14,61,Educational Services,56883.0,58222.0,59.0,59.0,1339.0,0.023540
2,22,Utilities,4301.0,4333.0,16.0,16.0,32.0,0.007440


### Matched-Sample Interpretation

Holding the locality sample constant materially changes the interpretation of several industry trends.

Transportation and Warehousing remains the strongest-growing sector, with matched employment increasing by approximately **33.5%** between 2019 and 2025. Arts, Entertainment, and Recreation, Construction, and Health Care and Social Assistance also show substantial growth within consistently observed localities.

The matched comparison also demonstrates why disclosure coverage must be controlled explicitly. For example, the apparent decline in Information employment becomes much smaller after restricting the analysis to localities observed in both years, while Construction shows considerably stronger growth than suggested by the unmatched totals.

These results indicate that changes in publication coverage can materially bias aggregate comparisons if the underlying geographic sample is allowed to change over time.

In [8]:
# Create a clean ranked table of matched 2019–2025 industry growth.
# ----------------------------------------------------------------
# This table will serve as the preferred statewide industry-growth
# benchmark because the locality sample is held constant across years.

matched_industry_growth_ranked = (
    matched_industry_comparison
    [
        [
            "industry_code",
            "industry_title",
            "matched_localities_2019",
            "matched_employment_2019",
            "matched_employment_2025",
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .rename(
        columns={
            "matched_localities_2019": "matched_localities"
        }
    )
    .sort_values(
        "employment_growth_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

matched_industry_growth_ranked

,industry_code,industry_title,matched_localities,matched_employment_2019,matched_employment_2025,employment_change,employment_growth_pct
0,48-49,Transportation and Warehousing,70.0,107633.0,143653.0,36020.0,0.334656
1,71,"Arts, Entertainment, and Recreation",91.0,53830.0,64790.0,10960.0,0.203604
2,99,Unclassified,49.0,5434.0,6349.0,915.0,0.168384
3,23,Construction,85.0,165261.0,187722.0,22461.0,0.135912
4,62,Health Care and Social Assistance,62.0,392163.0,444571.0,52408.0,0.133638
5,55,Management of Companies and Enterprises,54.0,72102.0,79546.0,7444.0,0.103243
6,53,Real Estate and Rental and Leasing,104.0,52744.0,57748.0,5004.0,0.094873
7,54,"Professional, Scientific, and Technical Services",84.0,390722.0,400870.0,10148.0,0.025972
8,61,Educational Services,59.0,56883.0,58222.0,1339.0,0.023540
9,22,Utilities,16.0,4301.0,4333.0,32.0,0.007440


### Statewide Industry Growth Benchmark

The matched-locality comparison provides the preferred statewide industry-growth benchmark for the project.

By holding the locality sample constant between 2019 and 2025, this ranking reduces the influence of changing disclosure coverage and provides a more defensible basis for comparing sector performance across Virginia.

In [9]:
# Save the matched statewide industry-growth ranking.
# ---------------------------------------------------
# This table will be reused in later visualizations and in the final
# project summary as the preferred statewide industry benchmark.

matched_industry_growth_path = (
    OUTPUT_TABLES
    / "matched_statewide_industry_growth_2019_2025.csv"
)

matched_industry_growth_ranked.to_csv(
    matched_industry_growth_path,
    index=False
)

print(
    f"Saved matched statewide industry-growth ranking to:\n"
    f"{matched_industry_growth_path}"
)

Saved matched statewide industry-growth ranking to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\matched_statewide_industry_growth_2019_2025.csv


### Answer to Research Question

**How has private-sector employment changed across Virginia industries from 2019 to 2025?**

Using a matched-locality sample to hold geographic coverage constant, the strongest employment growth occurred in **Transportation and Warehousing**, which increased by approximately **33.5%** between 2019 and 2025.

Other notable growth sectors included:

- **Arts, Entertainment, and Recreation:** +20.4%
- **Construction:** +13.6%
- **Health Care and Social Assistance:** +13.4%
- **Management of Companies and Enterprises:** +10.3%
- **Real Estate and Rental and Leasing:** +9.5%

The largest matched-sample declines occurred in:

- **Mining, Quarrying, and Oil and Gas Extraction:** -17.4%
- **Finance and Insurance:** -11.5%
- **Wholesale Trade:** -5.1%
- **Administrative and Support and Waste Management:** -4.7%
- **Agriculture, Forestry, Fishing and Hunting:** -4.5%

The analysis also demonstrates the importance of controlling for disclosure coverage. Several industries appeared to experience much larger declines when all published observations were compared directly, but those declines became substantially smaller after restricting the analysis to localities with published employment at both endpoints.

Overall, the results suggest that Virginia's strongest observed private-sector employment expansion during the period was concentrated in transportation and logistics, health-related services, construction, and selected service industries, while several traditional or mature sectors experienced modest contraction.

## 2. Locality-Level Employment Growth

Statewide industry trends provide the broader economic context, but regional policy analysis also requires identifying which individual Virginia localities are gaining or losing employment.

### Research Question

**Which Virginia counties and independent cities experienced the strongest private-sector employment growth between 2019 and 2025?**

To answer this, published private-sector employment is aggregated across industries within each locality. Because disclosure suppression varies across sectors and years, the initial locality totals are interpreted cautiously and will be followed by a matched-sector comparison that holds the industry composition constant across both endpoints.

In [10]:
# Aggregate published private-sector employment by locality and year.
# ------------------------------------------------------------------
# Suppressed industry observations remain missing and are therefore
# excluded from the summed published-employment total.
#
# These totals provide an initial view of locality-level employment
# change, but differences in sector publication coverage may affect
# comparisons across years.

locality_employment = (
    panel
    .groupby(
        ["year", "area_fips", "locality_name"],
        as_index=False
    )
    .agg(
        published_employment=("annual_avg_emplvl", "sum"),
        published_sectors=("annual_avg_emplvl", "count")
    )
)

locality_employment.head(20)

,year,area_fips,locality_name,published_employment,published_sectors
0,2019,51001,Accomack County,8386.0,13
1,2019,51003,Albemarle County,38633.0,20
2,2019,51005,Alleghany County,1563.0,8
3,2019,51007,Amelia County,1675.0,13
4,2019,51009,Amherst County,4592.0,14
5,2019,51011,Appomattox County,1801.0,13
6,2019,51013,Arlington County,140782.0,17
7,2019,51015,Augusta County,22754.0,18
8,2019,51017,Bath County,448.0,8
9,2019,51019,Bedford County,14483.0,17


### Matched-Sector Locality Comparison

Raw locality employment totals can be affected by changes in the number of industries with published data.

To create a more consistent comparison, each locality is evaluated using only industries with published employment in both 2019 and 2025.

This matched-sector approach holds the observable industry composition constant across the two endpoints and reduces the risk of interpreting changes in disclosure coverage as genuine local employment growth.

In [11]:
# Construct a matched-sector comparison within each locality.
# -----------------------------------------------------------
# Only locality × industry combinations with published employment
# in both 2019 and 2025 are retained.

matched_locality_endpoint_panel = (
    panel[
        panel["year"].isin([2019, 2025])
        & panel["annual_avg_emplvl"].notna()
    ]
    [
        [
            "year",
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title",
            "annual_avg_emplvl"
        ]
    ]
)

# Count the number of endpoint years available for each
# locality × industry combination.
matched_locality_series = (
    matched_locality_endpoint_panel
    .groupby(
        ["area_fips", "industry_code"]
    )["year"]
    .nunique()
    .reset_index(name="endpoint_years_available")
)

# Retain only industries observed at both endpoints.
matched_locality_series = matched_locality_series[
    matched_locality_series["endpoint_years_available"] == 2
]

matched_locality_endpoint_panel = (
    matched_locality_endpoint_panel
    .merge(
        matched_locality_series[
            ["area_fips", "industry_code"]
        ],
        on=["area_fips", "industry_code"],
        how="inner",
        validate="many_to_one"
    )
)

print(
    f"Matched locality-industry series: "
    f"{len(matched_locality_series):,}"
)

print(
    f"Matched endpoint observations: "
    f"{len(matched_locality_endpoint_panel):,}"
)

Matched locality-industry series: 1,459
Matched endpoint observations: 2,918


In [12]:
# Aggregate matched industry employment within each locality.
# -----------------------------------------------------------
# Because only industries observed in both 2019 and 2025 are retained,
# each locality is compared using a consistent set of published sectors.

matched_locality_employment = (
    matched_locality_endpoint_panel
    .groupby(
        ["year", "area_fips", "locality_name"],
        as_index=False
    )
    .agg(
        matched_employment=("annual_avg_emplvl", "sum"),
        matched_sectors=("industry_code", "nunique")
    )
)

# Reshape the two endpoint years so each locality occupies one row.
matched_locality_comparison = (
    matched_locality_employment
    .pivot(
        index=["area_fips", "locality_name"],
        columns="year",
        values=["matched_employment", "matched_sectors"]
    )
)

# Flatten the multi-level column names created by the pivot.
matched_locality_comparison.columns = [
    f"{measure}_{year}"
    for measure, year in matched_locality_comparison.columns
]

matched_locality_comparison = (
    matched_locality_comparison
    .reset_index()
)

# Calculate absolute and percentage employment change.
matched_locality_comparison["employment_change"] = (
    matched_locality_comparison["matched_employment_2025"]
    - matched_locality_comparison["matched_employment_2019"]
)

matched_locality_comparison["employment_growth_pct"] = (
    matched_locality_comparison["employment_change"]
    / matched_locality_comparison["matched_employment_2019"]
)

matched_locality_comparison.sort_values(
    "employment_growth_pct",
    ascending=False
).head(20)

,area_fips,locality_name,matched_employment_2019,matched_employment_2025,matched_sectors_2019,matched_sectors_2025,employment_change,employment_growth_pct
128,51800,Suffolk city,24625.0,35613.0,16.0,16.0,10988.0,0.446213
44,51091,Highland County,50.0,72.0,2.0,2.0,22.0,0.440000
18,51036,Charles City County,900.0,1268.0,6.0,6.0,368.0,0.408889
85,51179,Stafford County,25099.0,33133.0,11.0,11.0,8034.0,0.320092
48,51099,King George County,4342.0,5466.0,9.0,9.0,1124.0,0.258867
69,51145,Powhatan County,4513.0,5635.0,13.0,13.0,1122.0,0.248615
96,51520,Bristol city,4888.0,5974.0,11.0,11.0,1086.0,0.222177
112,51678,Lexington city,370.0,449.0,5.0,5.0,79.0,0.213514
83,51175,Southampton County,1040.0,1250.0,6.0,6.0,210.0,0.201923
33,51069,Frederick County,25563.0,30543.0,16.0,16.0,4980.0,0.194813


### Small-Base Effects in Locality Growth

Percentage growth can be highly sensitive to the starting employment base.

Several of the fastest-growing localities have relatively small matched employment totals in 2019. Although these growth rates are mathematically valid, they may overstate the economic significance of relatively small absolute employment gains.

For locality growth rankings, a minimum 2019 matched-employment threshold will therefore be applied. Absolute employment change will remain available separately so that smaller local economies are not removed from the underlying analysis.

In [13]:
# Apply a minimum baseline-employment threshold to locality growth rankings.
# ----------------------------------------------------------------------------
# The threshold reduces small-base distortions in percentage-growth comparisons
# while preserving the full dataset for other forms of analysis.

MIN_LOCALITY_BASE_EMPLOYMENT = 5_000

locality_growth_rank_eligible = (
    matched_locality_comparison[
        matched_locality_comparison["matched_employment_2019"]
        >= MIN_LOCALITY_BASE_EMPLOYMENT
    ]
    .copy()
)

print(
    f"Localities eligible for percentage-growth rankings: "
    f"{len(locality_growth_rank_eligible):,}"
)

print(
    f"Share of Virginia localities retained: "
    f"{len(locality_growth_rank_eligible) / len(matched_locality_comparison):.1%}"
)

Localities eligible for percentage-growth rankings: 65
Share of Virginia localities retained: 48.9%


### Locality Growth Ranking Eligibility

A minimum 2019 matched-employment base of **5,000 jobs** is used for the primary locality percentage-growth ranking.

This threshold retains 65 of Virginia's 133 counties and independent cities, or approximately **48.9%** of the statewide locality universe. The restriction is intentionally applied only to percentage-growth rankings, where small starting values can generate disproportionately large growth rates.

Smaller localities remain in the underlying dataset and may still be evaluated using absolute employment change and other regional indicators.

In [14]:
# Rank larger Virginia localities by matched 2019–2025 employment growth.
# -----------------------------------------------------------------------
# Only localities with at least 5,000 matched private-sector jobs in
# 2019 are included in this percentage-growth ranking to reduce
# small-base distortion.

top_locality_growth = (
    locality_growth_rank_eligible
    .sort_values(
        "employment_growth_pct",
        ascending=False
    )
    [
        [
            "area_fips",
            "locality_name",
            "matched_sectors_2019",
            "matched_employment_2019",
            "matched_employment_2025",
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .head(20)
)

top_locality_growth

,area_fips,locality_name,matched_sectors_2019,matched_employment_2019,matched_employment_2025,employment_change,employment_growth_pct
128,51800,Suffolk city,16.0,24625.0,35613.0,10988.0,0.446213
85,51179,Stafford County,11.0,25099.0,33133.0,8034.0,0.320092
33,51069,Frederick County,16.0,25563.0,30543.0,4980.0,0.194813
71,51149,Prince George County,10.0,5880.0,6970.0,1090.0,0.185374
104,51600,Fairfax city,12.0,16978.0,19876.0,2898.0,0.170691
36,51075,Goochland County,13.0,7518.0,8770.0,1252.0,0.166534
52,51107,Loudoun County,16.0,144351.0,168036.0,23685.0,0.164079
130,51820,Waynesboro city,11.0,6500.0,7377.0,877.0,0.134923
7,51015,Augusta County,16.0,22368.0,25050.0,2682.0,0.119903
72,51153,Prince William County,20.0,104954.0,114002.0,9048.0,0.086209


### Initial Locality Growth Interpretation

Among Virginia localities with at least 5,000 matched private-sector jobs in 2019, the strongest employment growth between 2019 and 2025 occurred in **Suffolk city**, where matched employment increased by approximately **44.6%**.

Other notable high-growth localities include **Stafford County (+32.0%)**, **Frederick County (+19.5%)**, **Fairfax city (+17.1%)**, and **Loudoun County (+16.4%)**.

Absolute job gains also provide important context. Loudoun County added approximately **23,685** matched jobs, while Suffolk city, Richmond city, Prince William County, Chesterfield County, and Stafford County each added several thousand jobs.

These results suggest that several of Virginia's fastest-growing local labor markets are located in Northern Virginia, the Richmond region, and the Hampton Roads corridor. Later industry-level analysis will help identify which sectors are driving these local gains.

In [15]:
# Identify the largest matched employment declines among larger localities.
# ------------------------------------------------------------------------
# The same 5,000-job baseline threshold is used so that positive and
# negative percentage-growth rankings remain directly comparable.

largest_locality_declines = (
    locality_growth_rank_eligible
    .sort_values(
        "employment_growth_pct",
        ascending=True
    )
    [
        [
            "area_fips",
            "locality_name",
            "matched_sectors_2019",
            "matched_employment_2019",
            "matched_employment_2025",
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .head(20)
)

largest_locality_declines

,area_fips,locality_name,matched_sectors_2019,matched_employment_2019,matched_employment_2025,employment_change,employment_growth_pct
45,51093,Isle of Wight County,14.0,8062.0,6696.0,-1366.0,-0.169437
82,51173,Smyth County,10.0,6796.0,5756.0,-1040.0,-0.153031
92,51195,Wise County,12.0,6092.0,5162.0,-930.0,-0.152659
116,51690,Martinsville city,10.0,5082.0,4353.0,-729.0,-0.143447
113,51680,Lynchburg city,15.0,44995.0,40205.0,-4790.0,-0.106456
81,51171,Shenandoah County,16.0,11449.0,10330.0,-1119.0,-0.097738
95,51510,Alexandria city,13.0,65248.0,59389.0,-5859.0,-0.089796
88,51185,Tazewell County,11.0,7503.0,6869.0,-634.0,-0.084500
73,51155,Pulaski County,9.0,10203.0,9370.0,-833.0,-0.081643
100,51570,Colonial Heights city,9.0,6691.0,6164.0,-527.0,-0.078763


### Locality Decline Interpretation

Among Virginia localities with at least 5,000 matched private-sector jobs in 2019, the largest employment declines occurred in **Isle of Wight County (-16.9%)**, **Smyth County (-15.3%)**, **Wise County (-15.3%)**, and **Martinsville city (-14.3%)**.

Larger urban economies also appear among the declining localities, including **Lynchburg city (-10.6%)**, **Alexandria city (-9.0%)**, **Hampton city (-7.8%)**, and **Charlottesville city (-7.0%)**.

The geographic diversity of these declines suggests that locality-level employment change cannot be explained by a single statewide pattern. Subsequent industry-level analysis is needed to determine whether local declines are concentrated in particular sectors or reflect broader changes in local economic structure.

### Answer to Research Question

**Which Virginia counties and independent cities experienced the strongest private-sector employment growth between 2019 and 2025?**

Using a matched-sector comparison and restricting the primary percentage-growth ranking to localities with at least 5,000 matched private-sector jobs in 2019, **Suffolk city recorded the strongest employment growth at approximately 44.6%**, followed by **Stafford County at 32.0%** and **Frederick County at 19.5%**.

Other notable high-growth localities included Fairfax city, Goochland County, Loudoun County, Augusta County, Prince William County, Richmond city, and Chesterfield County.

Absolute employment change provides additional context. **Loudoun County added approximately 23,685 matched jobs**, the largest gain among the leading growth localities, while Suffolk, Richmond, Prince William, Chesterfield, and Stafford also posted substantial job gains.

At the other end of the distribution, several localities experienced meaningful matched-employment declines, including Isle of Wight County, Smyth County, Wise County, Martinsville city, Lynchburg city, Alexandria city, Hampton city, and Charlottesville city.

Overall, employment growth between 2019 and 2025 was geographically uneven, with particularly strong gains in parts of Northern Virginia, the Richmond region, and the Hampton Roads corridor.

## 3. Industry Drivers of Local Employment Change

Locality-level growth rates show where employment expanded or contracted, but they do not explain **which industries produced those changes**.

### Research Question

**Which industries contributed most to employment growth or decline within Virginia's local economies between 2019 and 2025?**

To answer this question, employment change is calculated for each matched locality-industry series with published employment at both endpoints. These changes can then be ranked within each locality to identify the sectors responsible for the largest gains and losses.

In [17]:
# Calculate employment change for each matched locality × industry series.
# ------------------------------------------------------------------------
# Only locality × industry combinations with published employment in both
# 2019 and 2025 are included, ensuring comparable endpoint observations.

locality_industry_change = (
    matched_locality_endpoint_panel
    .pivot(
        index=[
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title"
        ],
        columns="year",
        values="annual_avg_emplvl"
    )
    .reset_index()
)

# Calculate the absolute number of jobs gained or lost between endpoints.
locality_industry_change["employment_change"] = (
    locality_industry_change[2025]
    - locality_industry_change[2019]
)

# Calculate percentage employment change for additional context.
locality_industry_change["employment_growth_pct"] = (
    locality_industry_change["employment_change"]
    / locality_industry_change[2019]
)

locality_industry_change[
    [
        "locality_name",
        "industry_title",
        2019,
        2025,
        "employment_change",
        "employment_growth_pct"
    ]
].head(20)

year,locality_name,industry_title,2019,2025,employment_change,employment_growth_pct
0,Accomack County,"Agriculture, Forestry, Fishing and Hunting",150.0,199.0,49.0,0.326667
1,Accomack County,Construction,391.0,349.0,-42.0,-0.107417
2,Accomack County,Manufacturing,3285.0,3229.0,-56.0,-0.017047
3,Accomack County,Wholesale Trade,240.0,167.0,-73.0,-0.304167
4,Accomack County,Retail Trade,1300.0,1176.0,-124.0,-0.095385
5,Accomack County,Information,81.0,96.0,15.0,0.185185
6,Accomack County,Finance and Insurance,145.0,199.0,54.0,0.372414
7,Accomack County,Real Estate and Rental and Leasing,100.0,148.0,48.0,0.480000
8,Accomack County,"Professional, Scientific, and Technical Services",932.0,905.0,-27.0,-0.028970
9,Accomack County,"Arts, Entertainment, and Recreation",104.0,173.0,69.0,0.663462


### Identifying Local Industry Drivers

Absolute employment change is used to identify the industries contributing most to each locality's growth or decline.

Percentage growth is useful for describing the pace of change, but absolute job change is more appropriate for identifying which industries had the largest effect on a locality's total employment trajectory.

In [18]:
# Rank industries within each locality by absolute employment change.
# ------------------------------------------------------------------
# Positive ranks identify the sectors adding the most jobs, while
# negative ranks identify the sectors losing the most jobs.

locality_industry_change["growth_rank_within_locality"] = (
    locality_industry_change
    .groupby("area_fips")["employment_change"]
    .rank(
        method="first",
        ascending=False
    )
)

locality_industry_change["decline_rank_within_locality"] = (
    locality_industry_change
    .groupby("area_fips")["employment_change"]
    .rank(
        method="first",
        ascending=True
    )
)

# Inspect the largest job-creating industry in each locality.
top_growth_driver_by_locality = (
    locality_industry_change[
        locality_industry_change["growth_rank_within_locality"] == 1
    ]
    [
        [
            "area_fips",
            "locality_name",
            "industry_title",
            2019,
            2025,
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .sort_values(
        "employment_change",
        ascending=False
    )
)

top_growth_driver_by_locality.head(20)

year,area_fips,locality_name,industry_title,2019,2025,employment_change,employment_growth_pct
293,51059,Fairfax County,Health Care and Social Assistance,60322.0,75707.0,15385.0,0.255048
1403,51800,Suffolk city,Transportation and Warehousing,2323.0,9944.0,7621.0,3.280672
562,51107,Loudoun County,Construction,17250.0,24219.0,6969.0,0.404000
454,51087,Henrico County,Transportation and Warehousing,3925.0,8876.0,4951.0,1.261401
798,51153,Prince William County,Health Care and Social Assistance,13894.0,18637.0,4743.0,0.341370
70,51013,Arlington County,Management of Companies and Enterprises,3159.0,7881.0,4722.0,1.494777
934,51179,Stafford County,Transportation and Warehousing,1581.0,6230.0,4649.0,2.940544
208,51041,Chesterfield County,Transportation and Warehousing,8629.0,12614.0,3985.0,0.461815
1099,51550,Chesapeake city,Transportation and Warehousing,4349.0,8157.0,3808.0,0.875604
1298,51710,Norfolk city,Health Care and Social Assistance,19792.0,23268.0,3476.0,0.175627


### Initial Industry-Driver Interpretation

The industries contributing most to local employment growth vary substantially across Virginia.

Health Care and Social Assistance is the largest employment-growth contributor in several major localities, including Fairfax County, Prince William County, Norfolk city, Alexandria city, Albemarle County, and Virginia Beach city.

Transportation and Warehousing is the dominant growth driver in several rapidly expanding local economies, including Suffolk city, Henrico County, Stafford County, Chesterfield County, and Chesapeake city.

Other localities show different growth engines. Construction is the largest contributor in Loudoun County, Management of Companies and Enterprises leads growth in Arlington County, and Administrative and Support Services contributes the most jobs in Richmond city.

These differences suggest that Virginia's regional employment growth is being generated by distinct local industry structures rather than a single statewide pattern.

In [19]:
# Identify the industry responsible for the largest employment decline
# within each Virginia locality.
# --------------------------------------------------------------------
# Absolute job loss is used rather than percentage decline because the
# objective is to identify which sector had the largest effect on the
# locality's overall employment trajectory.

top_decline_driver_by_locality = (
    locality_industry_change[
        locality_industry_change["decline_rank_within_locality"] == 1
    ]
    [
        [
            "area_fips",
            "locality_name",
            "industry_title",
            2019,
            2025,
            "employment_change",
            "employment_growth_pct"
        ]
    ]
    .sort_values(
        "employment_change",
        ascending=True
    )
)

top_decline_driver_by_locality.head(20)

year,area_fips,locality_name,industry_title,2019,2025,employment_change,employment_growth_pct
289,51059,Fairfax County,"Professional, Scientific, and Technical Services",159986.0,154006.0,-5980.0,-0.037378
455,51087,Henrico County,Finance and Insurance,16236.0,11163.0,-5073.0,-0.312454
71,51013,Arlington County,Administrative and Support and Waste Management,11431.0,6975.0,-4456.0,-0.389817
1055,51510,Alexandria city,"Professional, Scientific, and Technical Services",16222.0,12688.0,-3534.0,-0.217852
1098,51550,Chesapeake city,Retail Trade,15523.0,12904.0,-2619.0,-0.168717
1284,51700,Newport News city,Administrative and Support and Waste Management,6278.0,3821.0,-2457.0,-0.391367
214,51041,Chesterfield County,Administrative and Support and Waste Management,10979.0,8848.0,-2131.0,-0.194098
1420,51810,Virginia Beach city,Finance and Insurance,8241.0,6224.0,-2017.0,-0.244752
1297,51710,Norfolk city,Educational Services,3374.0,1746.0,-1628.0,-0.482513
1203,51650,Hampton city,Administrative and Support and Waste Management,3618.0,2124.0,-1494.0,-0.412935


### Initial Industry-Decline Interpretation

The industries contributing most to local employment decline also vary across Virginia, although several sectors appear repeatedly.

Administrative and Support and Waste Management is the largest declining sector in Arlington County, Newport News city, Chesterfield County, Hampton city, Lynchburg city, and Falls Church city.

Finance and Insurance is the largest source of job loss in Henrico County, Virginia Beach city, and Roanoke city, while Manufacturing is the primary declining sector in Isle of Wight County, Pulaski County, and Rockingham County.

Large losses in Professional, Scientific, and Technical Services are also visible in Fairfax County and Alexandria city.

These results reinforce the importance of examining local industry composition: localities can experience overall employment growth while simultaneously undergoing substantial contraction in individual sectors.

In [21]:
# Count how frequently each industry is the largest job-growth contributor
# across Virginia localities.
# -----------------------------------------------------------------------
# This helps distinguish isolated local growth stories from industries
# that repeatedly serve as the primary employment-growth engine across
# multiple regional economies.

growth_driver_frequency = (
    top_growth_driver_by_locality
    .groupby("industry_title")
    .size()
    .reset_index(name="localities_where_top_growth_driver")
    .sort_values(
        "localities_where_top_growth_driver",
        ascending=False
    )
    .reset_index(drop=True)
)

growth_driver_frequency

,industry_title,localities_where_top_growth_driver
0,Health Care and Social Assistance,21
1,Retail Trade,19
2,Manufacturing,16
3,Transportation and Warehousing,16
4,Construction,13
5,"Professional, Scientific, and Technical Services",10
6,Administrative and Support and Waste Management,8
7,Other Services,7
8,"Arts, Entertainment, and Recreation",6
9,Accommodation and Food Services,4


### Answer to Research Question

**Which industries contributed most to employment growth or decline within Virginia's local economies between 2019 and 2025?**

Employment growth was driven by different sectors across different local economies, but several industries emerged repeatedly as the largest contributors.

**Health Care and Social Assistance** was the most common top employment-growth driver, ranking first in 21 Virginia localities. **Retail Trade** was the leading growth contributor in 19 localities, while **Manufacturing** and **Transportation and Warehousing** each ranked first in 16 localities. Construction was the top growth contributor in 13 localities.

The size of individual contributions also varied substantially. Health Care and Social Assistance added more than 15,000 matched jobs in Fairfax County, while Transportation and Warehousing added more than 7,600 jobs in Suffolk city and more than 4,600 in Stafford County. Construction added nearly 7,000 jobs in Loudoun County.

Local employment losses were similarly sector-specific. Administrative and Support Services, Finance and Insurance, Manufacturing, and Professional, Scientific, and Technical Services were responsible for several of the largest locality-level declines.

Overall, Virginia's regional employment changes were produced by distinct local industry structures rather than a single statewide growth model. Health care was the most geographically widespread local growth engine, while transportation and logistics generated some of the largest concentrated job gains.

## 4. Regional Industry Specialization: Location Quotients

Employment growth shows which industries are expanding, but growth alone does not indicate whether an industry is unusually important to a local economy.

### Research Question

**Which industries are disproportionately concentrated in particular Virginia localities compared with the broader Virginia economy?**

A **Location Quotient (LQ)** measures the relative concentration of an industry within a locality compared with a reference economy.

The employment-based location quotient is calculated as:

$$
LQ_{i,r}
=
\frac{
\text{Industry } i \text{ employment in locality } r
\; / \;
\text{Total employment in locality } r
}{
\text{Industry } i \text{ employment in Virginia}
\; / \;
\text{Total employment in Virginia}
}
$$

Interpretation:

- **LQ = 1.0:** the industry's employment share matches the Virginia benchmark
- **LQ > 1.0:** the industry is more concentrated locally than statewide
- **LQ < 1.0:** the industry is less concentrated locally than statewide
- **LQ ≥ 1.25:** often treated as evidence of meaningful regional specialization

Because disclosure suppression can distort employment shares, location quotients will be calculated only from published employment observations and interpreted alongside data-coverage limitations.

In [22]:
# Create the 2025 published-employment dataset used for location quotients.
# ------------------------------------------------------------------------
# Only locality × industry observations with published employment values
# are included. Suppressed observations remain excluded because their
# employment levels are unavailable.

lq_2025 = (
    panel[
        (panel["year"] == 2025)
        & panel["annual_avg_emplvl"].notna()
    ]
    [
        [
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title",
            "annual_avg_emplvl"
        ]
    ]
    .copy()
)

print(f"Published 2025 locality-industry records: {len(lq_2025):,}")
print(f"Localities represented: {lq_2025['area_fips'].nunique()}")
print(f"Industries represented: {lq_2025['industry_code'].nunique()}")

Published 2025 locality-industry records: 1,569
Localities represented: 133
Industries represented: 20


### Method: Local Industry Employment Shares

The first component of the location quotient is the share of each locality's published private-sector employment represented by a given industry.

For each locality, total published employment is calculated across all industries with available 2025 employment data. Each industry's employment is then divided by that locality total.

In [23]:
# Calculate total published employment within each locality.
# ----------------------------------------------------------
# These totals are based only on industries with published employment
# because suppressed industry values are unavailable.

locality_totals_2025 = (
    lq_2025
    .groupby(
        ["area_fips", "locality_name"],
        as_index=False
    )
    .agg(
        locality_published_employment=(
            "annual_avg_emplvl",
            "sum"
        )
    )
)

# Attach the locality employment total to each industry observation.
lq_2025 = (
    lq_2025
    .merge(
        locality_totals_2025,
        on=["area_fips", "locality_name"],
        how="left",
        validate="many_to_one"
    )
)

# Calculate each industry's share of the locality's published employment.
lq_2025["local_industry_share"] = (
    lq_2025["annual_avg_emplvl"]
    / lq_2025["locality_published_employment"]
)

lq_2025[
    [
        "locality_name",
        "industry_title",
        "annual_avg_emplvl",
        "locality_published_employment",
        "local_industry_share"
    ]
].head(20)

,locality_name,industry_title,annual_avg_emplvl,locality_published_employment,local_industry_share
0,Accomack County,"Agriculture, Forestry, Fishing and Hunting",199.0,8289.0,0.024008
1,Accomack County,Construction,349.0,8289.0,0.042104
2,Accomack County,Manufacturing,3229.0,8289.0,0.389552
3,Accomack County,Wholesale Trade,167.0,8289.0,0.020147
4,Accomack County,Retail Trade,1176.0,8289.0,0.141875
5,Accomack County,Information,96.0,8289.0,0.011582
6,Accomack County,Finance and Insurance,199.0,8289.0,0.024008
7,Accomack County,Real Estate and Rental and Leasing,148.0,8289.0,0.017855
8,Accomack County,"Professional, Scientific, and Technical Services",905.0,8289.0,0.109181
9,Accomack County,"Arts, Entertainment, and Recreation",173.0,8289.0,0.020871


### Method: Virginia Industry Employment Shares

The second component of the location quotient is each industry's share of published private-sector employment across Virginia.

For each industry, published employment is summed across Virginia localities and divided by total published private-sector employment statewide.

These statewide industry shares serve as the benchmark against which each locality's industry concentration is compared.

In [24]:
# Calculate published employment by industry across Virginia.
# -----------------------------------------------------------
# These totals use the same 2025 published locality-level records
# used in the local-share calculation so that the numerator and
# denominator of the location quotient remain internally consistent.

state_industry_totals_2025 = (
    lq_2025
    .groupby(
        ["industry_code", "industry_title"],
        as_index=False
    )
    .agg(
        state_industry_published_employment=(
            "annual_avg_emplvl",
            "sum"
        )
    )
)

# Calculate total published private-sector employment across Virginia.
state_total_published_employment_2025 = (
    lq_2025["annual_avg_emplvl"].sum()
)

# Calculate each industry's share of statewide published employment.
state_industry_totals_2025["state_industry_share"] = (
    state_industry_totals_2025[
        "state_industry_published_employment"
    ]
    / state_total_published_employment_2025
)

state_industry_totals_2025.sort_values(
    "state_industry_share",
    ascending=False
)

,industry_code,industry_title,state_industry_published_employment,state_industry_share
15,62,Health Care and Social Assistance,459088.0,0.156256
11,54,"Professional, Scientific, and Technical Services",402656.0,0.137049
6,44-45,Retail Trade,383827.0,0.130640
17,72,Accommodation and Food Services,326759.0,0.111216
13,56,Administrative and Support and Waste Management,203045.0,0.069109
4,31-33,Manufacturing,197709.0,0.067293
3,23,Construction,190104.0,0.064704
7,48-49,Transportation and Warehousing,148300.0,0.050476
18,81,Other Services,130358.0,0.044369
9,52,Finance and Insurance,108803.0,0.037032


### Calculate Location Quotients

Each locality's industry employment share is divided by the corresponding Virginia industry employment share.

A location quotient greater than 1 indicates that the industry represents a larger share of local employment than it does statewide. Values of 1.25 or greater are treated as evidence of meaningful regional specialization for this analysis.

In [25]:
# Attach statewide industry shares to each locality-industry observation.
# ----------------------------------------------------------------------
# The join is many-to-one because many locality observations correspond
# to one statewide industry benchmark.

lq_2025 = (
    lq_2025
    .merge(
        state_industry_totals_2025[
            [
                "industry_code",
                "state_industry_share"
            ]
        ],
        on="industry_code",
        how="left",
        validate="many_to_one"
    )
)

# Calculate the employment-based location quotient.
lq_2025["location_quotient"] = (
    lq_2025["local_industry_share"]
    / lq_2025["state_industry_share"]
)

# Flag industries with meaningful regional specialization.
LQ_SPECIALIZATION_THRESHOLD = 1.25

lq_2025["is_specialized"] = (
    lq_2025["location_quotient"]
    >= LQ_SPECIALIZATION_THRESHOLD
)

lq_2025[
    [
        "locality_name",
        "industry_title",
        "local_industry_share",
        "state_industry_share",
        "location_quotient",
        "is_specialized"
    ]
].head(20)

,locality_name,industry_title,local_industry_share,state_industry_share,location_quotient,is_specialized
0,Accomack County,"Agriculture, Forestry, Fishing and Hunting",0.024008,0.001347,17.825566,True
1,Accomack County,Construction,0.042104,0.064704,0.650714,False
2,Accomack County,Manufacturing,0.389552,0.067293,5.788925,True
3,Accomack County,Wholesale Trade,0.020147,0.021690,0.928858,False
4,Accomack County,Retail Trade,0.141875,0.130640,1.085996,False
5,Accomack County,Information,0.011582,0.014622,0.792088,False
6,Accomack County,Finance and Insurance,0.024008,0.037032,0.648289,False
7,Accomack County,Real Estate and Rental and Leasing,0.017855,0.019764,0.903402,False
8,Accomack County,"Professional, Scientific, and Technical Services",0.109181,0.137049,0.796656,False
9,Accomack County,"Arts, Entertainment, and Recreation",0.020871,0.022076,0.945436,False


### Minimum Employment Requirement for Specialization Rankings

Location quotients can become very large when an industry represents only a small share of statewide employment.

A high LQ may therefore identify genuine regional specialization, but it can also arise from a relatively small number of local jobs in an industry with a very small statewide employment base.

For the primary specialization rankings, an industry must therefore satisfy both conditions:

- **Location Quotient ≥ 1.25**
- **At least 100 published local employees**

The employment threshold is used only for headline specialization rankings. All calculated location quotients remain available in the underlying analytical dataset.

In [26]:
# Define the minimum local employment level required for the primary
# specialization ranking.
#
# This reduces the influence of extremely high LQs generated by very
# small local employment cells while retaining all observations in
# the underlying LQ dataset.

MIN_LQ_EMPLOYMENT = 100

lq_2025["ranking_eligible_specialization"] = (
    (lq_2025["location_quotient"] >= LQ_SPECIALIZATION_THRESHOLD)
    & (lq_2025["annual_avg_emplvl"] >= MIN_LQ_EMPLOYMENT)
)

print(
    f"Specialized locality-industry observations "
    f"(LQ >= {LQ_SPECIALIZATION_THRESHOLD}): "
    f"{lq_2025['is_specialized'].sum():,}"
)

print(
    f"Ranking-eligible specializations "
    f"(LQ >= {LQ_SPECIALIZATION_THRESHOLD} and employment >= {MIN_LQ_EMPLOYMENT}): "
    f"{lq_2025['ranking_eligible_specialization'].sum():,}"
)

Specialized locality-industry observations (LQ >= 1.25): 648
Ranking-eligible specializations (LQ >= 1.25 and employment >= 100): 542


### Specialization Ranking Eligibility

In 2025, 648 locality-industry observations have a location quotient of at least 1.25.

After requiring at least 100 published local employees, 542 observations remain eligible for the primary specialization rankings. This retains most identified specializations while reducing the influence of very small employment cells that can generate disproportionately large location quotients.

In [27]:
# Rank the strongest regional industry specializations in Virginia.
# -----------------------------------------------------------------
# Only observations meeting both the LQ threshold and minimum local
# employment requirement are included in the headline ranking.

top_specializations_2025 = (
    lq_2025[
        lq_2025["ranking_eligible_specialization"]
    ]
    .sort_values(
        "location_quotient",
        ascending=False
    )
    [
        [
            "area_fips",
            "locality_name",
            "industry_title",
            "annual_avg_emplvl",
            "local_industry_share",
            "state_industry_share",
            "location_quotient"
        ]
    ]
    .head(20)
)

top_specializations_2025

,area_fips,locality_name,industry_title,annual_avg_emplvl,local_industry_share,state_industry_share,location_quotient
729,51131,Northampton County,"Agriculture, Forestry, Fishing and Hunting",733.0,0.279238,0.001347,207.332345
939,51167,Russell County,"Mining, Quarrying, and Oil and Gas Extraction",181.0,0.066642,0.000339,196.386710
621,51109,Louisa County,Utilities,1016.0,0.233886,0.002143,109.160772
1021,51183,Sussex County,"Agriculture, Forestry, Fishing and Hunting",155.0,0.132027,0.001347,98.029321
739,51133,Northumberland County,"Agriculture, Forestry, Fishing and Hunting",153.0,0.095565,0.001347,70.956550
1073,51193,Westmoreland County,"Agriculture, Forestry, Fishing and Hunting",186.0,0.087447,0.001347,64.928870
1102,51197,Wythe County,"Mining, Quarrying, and Oil and Gas Extraction",136.0,0.017623,0.000339,51.934230
516,51091,Highland County,Information,108.0,0.551020,0.014622,37.685299
421,51075,Goochland County,"Mining, Quarrying, and Oil and Gas Extraction",108.0,0.012315,0.000339,36.290040
777,51141,Patrick County,"Agriculture, Forestry, Fishing and Hunting",105.0,0.040635,0.001347,30.170964


### Dominant Industry Specialization by Locality

A statewide ranking highlights the most extreme specialization values, but it can be dominated by a small number of industries with very low statewide employment shares.

To provide a more geographically balanced view, the analysis also identifies the highest ranking-eligible location quotient within each Virginia locality.

This reveals the industry that most strongly distinguishes each local economy from the statewide employment structure.

In [28]:
# Identify the strongest ranking-eligible specialization in each locality.
# -----------------------------------------------------------------------
# Only industries meeting both the LQ threshold and minimum-employment
# requirement are considered for the locality-level specialization ranking.

top_specialization_by_locality = (
    lq_2025[
        lq_2025["ranking_eligible_specialization"]
    ]
    .sort_values(
        ["area_fips", "location_quotient"],
        ascending=[True, False]
    )
    .groupby(
        ["area_fips", "locality_name"],
        as_index=False
    )
    .first()
)

top_specialization_by_locality[
    [
        "area_fips",
        "locality_name",
        "industry_title",
        "annual_avg_emplvl",
        "local_industry_share",
        "location_quotient"
    ]
].head(20)


,area_fips,locality_name,industry_title,annual_avg_emplvl,local_industry_share,location_quotient
0,51001,Accomack County,"Agriculture, Forestry, Fishing and Hunting",199.0,0.024008,17.825566
1,51003,Albemarle County,"Agriculture, Forestry, Fishing and Hunting",719.0,0.017668,13.118703
2,51005,Alleghany County,Health Care and Social Assistance,830.0,0.470255,3.009511
3,51007,Amelia County,Construction,444.0,0.367854,5.685163
4,51009,Amherst County,Manufacturing,1386.0,0.352582,5.239528
5,51011,Appomattox County,Construction,421.0,0.219499,3.392350
6,51013,Arlington County,"Professional, Scientific, and Technical Services",48108.0,0.354546,2.587003
7,51015,Augusta County,Manufacturing,6744.0,0.269222,4.000754
8,51017,Bath County,"Professional, Scientific, and Technical Services",125.0,0.578704,4.222606
9,51019,Bedford County,Wholesale Trade,707.0,0.046233,2.131523


In [29]:
# Count how often each industry is the strongest specialization
# across Virginia localities.
# ----------------------------------------------------------------
# This reveals which sectors most frequently define a locality's
# distinctive employment structure relative to the statewide economy.

top_specialization_frequency = (
    top_specialization_by_locality
    .groupby("industry_title")
    .size()
    .reset_index(name="localities_where_top_specialization")
    .sort_values(
        "localities_where_top_specialization",
        ascending=False
    )
    .reset_index(drop=True)
)

top_specialization_frequency

,industry_title,localities_where_top_specialization
0,Manufacturing,33
1,Wholesale Trade,13
2,Construction,10
3,"Agriculture, Forestry, Fishing and Hunting",9
4,Retail Trade,9
5,Health Care and Social Assistance,8
6,Educational Services,7
7,Transportation and Warehousing,7
8,Utilities,7
9,"Arts, Entertainment, and Recreation",6


### Answer to Research Question

**Which industries are disproportionately concentrated in particular Virginia localities compared with the broader Virginia economy?**

Location quotient analysis reveals substantial variation in industry specialization across Virginia.

**Manufacturing is the most common dominant specialization**, ranking as the strongest qualifying industry concentration in 33 Virginia localities. Wholesale Trade is the top specialization in 13 localities, followed by Construction in 10 and Agriculture in 9.

Several local economies exhibit particularly strong specialization. Examples include:

- Northampton County in Agriculture, Forestry, Fishing and Hunting
- Russell County in Mining, Quarrying, and Oil and Gas Extraction
- Louisa County in Utilities
- Bland County in Manufacturing
- Dinwiddie County in Transportation and Warehousing
- Arlington County in Professional, Scientific, and Technical Services

These results show that Virginia's regional economies have markedly different industrial identities. Manufacturing remains the most geographically widespread source of specialization, while agriculture, utilities, mining, logistics, and professional services define more concentrated regional clusters.

Location quotients should be interpreted as measures of **relative concentration rather than industry size or growth**. A highly specialized industry may be economically important to a locality even if it is not growing rapidly, while a fast-growing industry may still represent a relatively small share of the local economy.

## 5. Shift-Share Analysis

Employment growth alone cannot determine whether a local industry performed well because of favorable statewide conditions or because the locality itself exhibited unusually strong performance.

### Research Question

**Are Virginia locality-industry employment changes explained primarily by statewide economic growth, industry-specific trends, or locally competitive performance?**

Shift-share analysis decomposes employment change into three components:

1. **Statewide Growth Effect**  
   The employment change expected if the local industry had grown at the overall Virginia private-sector growth rate.

2. **Industry Mix Effect**  
   The additional gain or loss associated with the industry's statewide performance relative to the overall Virginia economy.

3. **Regional Competitive Effect**  
   The portion of employment change associated with the local industry's performance relative to the same industry statewide.

For locality $r$ and industry $i$:

$$
SGE_{i,r} = E_{i,r,2019} \times g_{VA}
$$

$$
IME_{i,r} = E_{i,r,2019} \times (g_{i,VA} - g_{VA})
$$

$$
RCE_{i,r} = E_{i,r,2019} \times (g_{i,r} - g_{i,VA})
$$

where:

- $E_{i,r,2019}$ = local industry employment in 2019
- $g_{VA}$ = overall Virginia employment growth rate
- $g_{i,VA}$ = statewide growth rate for industry $i$
- $g_{i,r}$ = local growth rate for industry $i$

The components sum to the observed employment change:

$$
\Delta E_{i,r} = SGE_{i,r} + IME_{i,r} + RCE_{i,r}
$$

Because disclosure coverage changes over time, the analysis uses the previously constructed **matched locality-industry sample**, ensuring that the same underlying series are represented at both endpoints.

In [30]:
# Calculate overall Virginia employment growth within the matched sample.
# ----------------------------------------------------------------------
# The same 1,459 locality × industry series are represented in both
# 2019 and 2025, so the statewide benchmark is not affected by changes
# in endpoint publication coverage.

state_matched_employment = (
    matched_locality_endpoint_panel
    .groupby("year", as_index=False)
    .agg(
        matched_employment=("annual_avg_emplvl", "sum")
    )
)

# Extract the two endpoint totals.
va_employment_2019 = (
    state_matched_employment
    .loc[
        state_matched_employment["year"] == 2019,
        "matched_employment"
    ]
    .iloc[0]
)

va_employment_2025 = (
    state_matched_employment
    .loc[
        state_matched_employment["year"] == 2025,
        "matched_employment"
    ]
    .iloc[0]
)

# Calculate the overall matched-sample Virginia employment growth rate.
va_growth_rate = (
    (va_employment_2025 - va_employment_2019)
    / va_employment_2019
)

print(f"Matched Virginia employment, 2019: {va_employment_2019:,.0f}")
print(f"Matched Virginia employment, 2025: {va_employment_2025:,.0f}")
print(f"Matched Virginia employment growth: {va_growth_rate:.2%}")

Matched Virginia employment, 2019: 2,813,166
Matched Virginia employment, 2025: 2,895,583
Matched Virginia employment growth: 2.93%


### Statewide Growth Benchmark

Across the matched locality-industry sample, Virginia private-sector employment increased from approximately **2.81 million jobs in 2019 to 2.90 million jobs in 2025**, representing growth of **2.93%**.

This 2.93% rate serves as the statewide growth benchmark in the shift-share decomposition. The industry-mix and regional competitive effects will measure how each locality-industry combination performed relative to this overall trend.

In [32]:
# Calculate matched-sample statewide growth rates by industry.
# -----------------------------------------------------------
# These industry-specific growth rates provide the benchmark needed
# to separate broad sector trends from locality-specific performance.

state_industry_growth_rates = (
    matched_industry_comparison[
        [
            "industry_code",
            "industry_title",
            "matched_employment_2019",
            "matched_employment_2025",
            "employment_growth_pct"
        ]
    ]
    .rename(
        columns={
            "employment_growth_pct": "state_industry_growth_rate"
        }
    )
    .copy()
)

state_industry_growth_rates.sort_values(
    "state_industry_growth_rate",
    ascending=False
)

,industry_code,industry_title,matched_employment_2019,matched_employment_2025,state_industry_growth_rate
7,48-49,Transportation and Warehousing,107633.0,143653.0,0.334656
16,71,"Arts, Entertainment, and Recreation",53830.0,64790.0,0.203604
19,99,Unclassified,5434.0,6349.0,0.168384
3,23,Construction,165261.0,187722.0,0.135912
15,62,Health Care and Social Assistance,392163.0,444571.0,0.133638
12,55,Management of Companies and Enterprises,72102.0,79546.0,0.103243
10,53,Real Estate and Rental and Leasing,52744.0,57748.0,0.094873
11,54,"Professional, Scientific, and Technical Services",390722.0,400870.0,0.025972
14,61,Educational Services,56883.0,58222.0,0.023540
2,22,Utilities,4301.0,4333.0,0.007440


### Construct the Shift-Share Analytical Table

Each matched locality-industry series is combined with the corresponding statewide industry growth rate.

This creates the common analytical table needed to calculate the statewide growth effect, industry mix effect, and regional competitive effect for every consistently observed locality-industry combination.

In [33]:
# Combine local employment change with statewide industry growth benchmarks.
# ------------------------------------------------------------------------
# Each locality × industry series receives the matched-sample growth rate
# for the same industry across Virginia.

shift_share = (
    locality_industry_change
    .merge(
        state_industry_growth_rates[
            [
                "industry_code",
                "state_industry_growth_rate"
            ]
        ],
        on="industry_code",
        how="left",
        validate="many_to_one"
    )
)

# Calculate each locality-industry's observed employment growth rate.
shift_share["local_industry_growth_rate"] = (
    (shift_share[2025] - shift_share[2019])
    / shift_share[2019]
)

# Confirm that every locality-industry series received a statewide benchmark.
print(
    "Records missing statewide industry growth rate:",
    shift_share["state_industry_growth_rate"].isna().sum()
)

shift_share[
    [
        "locality_name",
        "industry_title",
        2019,
        2025,
        "local_industry_growth_rate",
        "state_industry_growth_rate"
    ]
].head(20)

Records missing statewide industry growth rate: 0


,locality_name,industry_title,2019,2025,local_industry_growth_rate,state_industry_growth_rate
0,Accomack County,"Agriculture, Forestry, Fishing and Hunting",150.0,199.0,0.326667,-0.045066
1,Accomack County,Construction,391.0,349.0,-0.107417,0.135912
2,Accomack County,Manufacturing,3285.0,3229.0,-0.017047,-0.032379
3,Accomack County,Wholesale Trade,240.0,167.0,-0.304167,-0.050562
4,Accomack County,Retail Trade,1300.0,1176.0,-0.095385,-0.044432
5,Accomack County,Information,81.0,96.0,0.185185,-0.026894
6,Accomack County,Finance and Insurance,145.0,199.0,0.372414,-0.114806
7,Accomack County,Real Estate and Rental and Leasing,100.0,148.0,0.480000,0.094873
8,Accomack County,"Professional, Scientific, and Technical Services",932.0,905.0,-0.028970,0.025972
9,Accomack County,"Arts, Entertainment, and Recreation",104.0,173.0,0.663462,0.203604


### Calculate Shift-Share Components

The observed employment change for each locality-industry series is decomposed into three additive components:

- **Statewide Growth Effect:** expected change if the local industry followed overall Virginia employment growth.
- **Industry Mix Effect:** additional change associated with the industry's statewide performance relative to overall Virginia growth.
- **Regional Competitive Effect:** additional change associated with the locality outperforming or underperforming the same industry statewide.

A positive regional competitive effect indicates that the local industry performed better than its statewide industry benchmark, while a negative value indicates relative underperformance.

In [34]:
# Calculate the three shift-share components.
# -------------------------------------------
# All effects are expressed as numbers of jobs so that they can be
# interpreted directly and summed back to the observed employment change.

# 1. Statewide Growth Effect:
# Expected local job change if the industry simply followed the
# overall matched Virginia private-sector growth rate.
shift_share["statewide_growth_effect"] = (
    shift_share[2019]
    * va_growth_rate
)

# 2. Industry Mix Effect:
# Captures whether the industry was growing faster or slower statewide
# than the Virginia economy overall.
shift_share["industry_mix_effect"] = (
    shift_share[2019]
    * (
        shift_share["state_industry_growth_rate"]
        - va_growth_rate
    )
)

# 3. Regional Competitive Effect:
# Captures local performance relative to the same industry statewide.
shift_share["regional_competitive_effect"] = (
    shift_share[2019]
    * (
        shift_share["local_industry_growth_rate"]
        - shift_share["state_industry_growth_rate"]
    )
)

# Observed employment change from 2019 to 2025.
shift_share["observed_employment_change"] = (
    shift_share[2025]
    - shift_share[2019]
)

shift_share[
    [
        "locality_name",
        "industry_title",
        "observed_employment_change",
        "statewide_growth_effect",
        "industry_mix_effect",
        "regional_competitive_effect"
    ]
].head(20)

,locality_name,industry_title,observed_employment_change,statewide_growth_effect,industry_mix_effect,regional_competitive_effect
0,Accomack County,"Agriculture, Forestry, Fishing and Hunting",49.0,4.394533,-11.154373,55.759840
1,Accomack County,Construction,-42.0,11.455082,41.686621,-95.141703
2,Accomack County,Manufacturing,-56.0,96.240266,-202.604584,50.364318
3,Accomack County,Wholesale Trade,-73.0,7.031252,-19.166171,-60.865081
4,Accomack County,Retail Trade,-124.0,38.085950,-95.846970,-66.238980
5,Accomack County,Information,15.0,2.373048,-4.551487,17.178439
6,Accomack County,Finance and Insurance,54.0,4.248048,-20.894863,70.646815
7,Accomack County,Real Estate and Rental and Leasing,48.0,2.929688,6.557647,38.512665
8,Accomack County,"Professional, Scientific, and Technical Services",-27.0,27.304697,-3.098391,-51.206305
9,Accomack County,"Arts, Entertainment, and Recreation",69.0,3.046876,18.127934,47.825190


In [35]:
# Validate the shift-share identity.
# ----------------------------------
# For every locality × industry series, the three shift-share components
# should sum to the observed employment change, subject only to very small
# floating-point rounding differences.

shift_share["decomposition_sum"] = (
    shift_share["statewide_growth_effect"]
    + shift_share["industry_mix_effect"]
    + shift_share["regional_competitive_effect"]
)

shift_share["decomposition_error"] = (
    shift_share["observed_employment_change"]
    - shift_share["decomposition_sum"]
)

shift_share["decomposition_error"].abs().describe()

count    1.459000e+03
mean     1.383369e-14
std      5.508207e-14
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      3.552714e-15
max      9.094947e-13
Name: decomposition_error, dtype: float64

### Shift-Share Validation

The shift-share decomposition reproduces observed employment change with effectively zero error.

The maximum absolute decomposition difference is approximately \(9 \times 10^{-13}\) jobs, which reflects normal floating-point precision rather than a substantive discrepancy.

This confirms that the statewide growth, industry mix, and regional competitive components have been calculated consistently.

In [36]:
# Identify the strongest positive regional competitive effects.
# -------------------------------------------------------------
# A large positive competitive effect means that the local industry
# added substantially more employment than would have been expected
# based on the statewide performance of that same industry.

top_competitive_advantages = (
    shift_share
    .sort_values(
        "regional_competitive_effect",
        ascending=False
    )
    [
        [
            "locality_name",
            "industry_title",
            2019,
            2025,
            "observed_employment_change",
            "state_industry_growth_rate",
            "local_industry_growth_rate",
            "regional_competitive_effect"
        ]
    ]
    .head(20)
)

top_competitive_advantages

,locality_name,industry_title,2019,2025,observed_employment_change,state_industry_growth_rate,local_industry_growth_rate,regional_competitive_effect
293,Fairfax County,Health Care and Social Assistance,60322.0,75707.0,15385.0,0.133638,0.255048,7323.669951
1403,Suffolk city,Transportation and Warehousing,2323.0,9944.0,7621.0,0.334656,3.280672,6843.594743
562,Loudoun County,Construction,17250.0,24219.0,6969.0,0.135912,0.404000,4624.513097
70,Arlington County,Management of Companies and Enterprises,3159.0,7881.0,4722.0,0.103243,1.494777,4395.856537
291,Fairfax County,Administrative and Support and Waste Management,43114.0,45451.0,2337.0,-0.047378,0.054205,4379.663027
934,Stafford County,Transportation and Warehousing,1581.0,6230.0,4649.0,0.334656,2.940544,4119.909294
569,Loudoun County,"Professional, Scientific, and Technical Services",21149.0,25626.0,4477.0,0.025972,0.211688,3927.709067
573,Loudoun County,Health Care and Social Assistance,13647.0,19235.0,5588.0,0.133638,0.409467,3764.238003
454,Henrico County,Transportation and Warehousing,3925.0,8876.0,4951.0,0.334656,1.261401,3637.476267
1357,Richmond city,Administrative and Support and Waste Management,9170.0,12240.0,3070.0,-0.047378,0.334787,3504.457948


### Positive Regional Competitive Effects

Several locality-industry combinations substantially outperformed their corresponding statewide industry trends.

Fairfax County's Health Care and Social Assistance sector shows the largest positive regional competitive effect, contributing approximately **7,324 more jobs** than would have been expected based on statewide health-care performance alone.

Suffolk city's Transportation and Warehousing sector also stands out, with an estimated competitive effect of approximately **6,844 jobs**. Although Transportation and Warehousing grew strongly across Virginia, Suffolk's local employment growth substantially exceeded the statewide sector trend.

Other notable positive competitive effects include Construction in Loudoun County, Management of Companies and Enterprises in Arlington County, Transportation and Warehousing in Stafford County, and Professional, Scientific, and Technical Services in Loudoun County.

These results identify local industries whose performance cannot be explained solely by statewide economic growth or favorable industry trends. They represent candidates for deeper investigation into local competitive advantages, investment patterns, infrastructure, workforce conditions, or other place-specific factors.

In [37]:
# Identify the strongest negative regional competitive effects.
# -------------------------------------------------------------
# A large negative competitive effect means that the local industry
# performed substantially worse than the same industry across Virginia,
# after accounting for statewide economic and industry-specific trends.

largest_competitive_disadvantages = (
    shift_share
    .sort_values(
        "regional_competitive_effect",
        ascending=True
    )
    [
        [
            "locality_name",
            "industry_title",
            2019,
            2025,
            "observed_employment_change",
            "state_industry_growth_rate",
            "local_industry_growth_rate",
            "regional_competitive_effect"
        ]
    ]
    .head(20)
)

largest_competitive_disadvantages

,locality_name,industry_title,2019,2025,observed_employment_change,state_industry_growth_rate,local_industry_growth_rate,regional_competitive_effect
289,Fairfax County,"Professional, Scientific, and Technical Services",159986.0,154006.0,-5980.0,0.025972,-0.037378,-10135.225270
461,Henrico County,Health Care and Social Assistance,28355.0,27470.0,-885.0,0.133638,-0.031211,-4674.314239
1055,Alexandria city,"Professional, Scientific, and Technical Services",16222.0,12688.0,-3534.0,0.025972,-0.217852,-3955.324768
71,Arlington County,Administrative and Support and Waste Management,11431.0,6975.0,-4456.0,-0.047378,-0.389817,-3914.419978
455,Henrico County,Finance and Insurance,16236.0,11163.0,-5073.0,-0.114806,-0.312454,-3209.015948
565,Loudoun County,Transportation and Warehousing,11877.0,13337.0,1460.0,0.334656,0.122927,-2514.706085
1416,Virginia Beach city,Construction,10324.0,9264.0,-1060.0,0.135912,-0.102673,-2463.158422
1331,Portsmouth city,Health Care and Social Assistance,7082.0,5637.0,-1445.0,0.133638,-0.204038,-2391.426501
285,Fairfax County,Transportation and Warehousing,10100.0,11252.0,1152.0,0.334656,0.114059,-2228.022855
1284,Newport News city,Administrative and Support and Waste Management,6278.0,3821.0,-2457.0,-0.047378,-0.391367,-2159.559761


### Negative Regional Competitive Effects

Several locality-industry combinations substantially underperformed the corresponding statewide industry trend.

The largest negative regional competitive effect occurs in **Fairfax County's Professional, Scientific, and Technical Services sector**, where local employment declined while the same industry grew modestly across the matched Virginia sample. The resulting competitive effect is approximately **-10,135 jobs**.

Other substantial negative competitive effects are observed in Health Care and Social Assistance in Henrico County, Professional and Technical Services in Alexandria city, Administrative and Support Services in Arlington County, and Finance and Insurance in Henrico County.

A negative competitive effect does not necessarily imply an absolute employment decline. In some cases, local employment still increased but grew more slowly than the same industry statewide. The competitive component therefore measures **relative local performance**, not simply whether employment increased or decreased.

In [38]:
# Aggregate regional competitive effects across industries within each locality.
# ------------------------------------------------------------------------------
# Summing the industry-level competitive effects provides a locality-level
# estimate of how much employment change was associated with local performance
# beyond statewide growth and industry-specific trends.

locality_competitive_effects = (
    shift_share
    .groupby(
        ["area_fips", "locality_name"],
        as_index=False
    )
    .agg(
        total_regional_competitive_effect=(
            "regional_competitive_effect",
            "sum"
        ),
        observed_employment_change=(
            "observed_employment_change",
            "sum"
        )
    )
    .sort_values(
        "total_regional_competitive_effect",
        ascending=False
    )
)

locality_competitive_effects.head(20)

,area_fips,locality_name,total_regional_competitive_effect,observed_employment_change
52,51107,Loudoun County,16428.778522,23685.0
128,51800,Suffolk city,9601.941198,10988.0
85,51179,Stafford County,6880.464922,8034.0
72,51153,Prince William County,5387.997365,9048.0
124,51760,Richmond city,4683.077992,9325.0
20,51041,Chesterfield County,4069.106324,8800.0
33,51069,Frederick County,3893.993133,4980.0
104,51600,Fairfax city,2625.329985,2898.0
99,51550,Chesapeake city,1971.303032,4384.0
7,51015,Augusta County,1108.853215,2682.0


### Locality-Level Competitive Performance

Aggregating regional competitive effects across industries provides a locality-level measure of employment performance beyond statewide growth and industry composition.

**Loudoun County** shows the strongest positive overall competitive effect, with approximately **16,429 jobs** associated with local performance beyond what statewide and industry trends would predict.

Other localities with strong positive competitive effects include **Suffolk city, Stafford County, Prince William County, Richmond city, Chesterfield County, and Frederick County**.

These results suggest that employment growth in these localities was not driven solely by favorable statewide sector trends. Instead, their local industries collectively outperformed comparable industries elsewhere in Virginia.

In [39]:
# Identify localities with the strongest negative overall competitive effects.
# ----------------------------------------------------------------------------
# Negative values indicate that the locality's industries collectively
# underperformed their corresponding statewide industry benchmarks.

locality_competitive_effects.tail(20).sort_values(
    "total_regional_competitive_effect",
    ascending=True
)

,area_fips,locality_name,total_regional_competitive_effect,observed_employment_change
95,51510,Alexandria city,-7248.483620,-5859.0
113,51680,Lynchburg city,-5601.513916,-4790.0
28,51059,Fairfax County,-5017.516514,8571.0
129,51810,Virginia Beach city,-4695.995058,-1757.0
6,51013,Arlington County,-4341.499395,743.0
42,51087,Henrico County,-4071.675483,19.0
109,51650,Hampton city,-3503.488433,-2548.0
76,51161,Roanoke County,-3079.070176,-1991.0
125,51770,Roanoke city,-2685.406308,410.0
98,51540,Charlottesville city,-2303.712283,-2017.0


### Negative Locality-Level Competitive Performance

Several Virginia localities show substantial negative regional competitive effects, indicating that their industries collectively underperformed comparable industries across the state.

**Alexandria city** has the largest negative overall competitive effect, at approximately **-7,248 jobs**, followed by **Lynchburg city (-5,602)**, **Fairfax County (-5,018)**, **Virginia Beach city (-4,696)**, and **Arlington County (-4,341)**.

Importantly, a negative competitive effect does not necessarily imply overall employment decline. Fairfax County, Arlington County, Henrico County, Roanoke city, and Norfolk city all recorded positive or near-flat observed employment change while still exhibiting negative competitive effects.

This distinction illustrates the value of shift-share analysis: a locality can add jobs because it has exposure to favorable statewide or industry trends while simultaneously underperforming those same industry benchmarks locally.

### Answer to Research Question

**Are Virginia locality-industry employment changes explained primarily by statewide economic growth, industry-specific trends, or locally competitive performance?**

Shift-share analysis shows that local employment change reflects a combination of all three forces, but the importance of each component varies substantially across Virginia.

The strongest positive regional competitive effects are concentrated in localities such as **Loudoun County, Suffolk city, Stafford County, Prince William County, Richmond city, Chesterfield County, and Frederick County**. These places added substantially more employment than would have been expected from statewide growth and industry composition alone.

At the industry level, particularly strong competitive effects appear in Fairfax County's Health Care sector, Suffolk city's Transportation and Warehousing sector, Loudoun County's Construction and Professional Services sectors, Arlington County's Management of Companies sector, and Stafford County's Transportation and Warehousing sector.

Conversely, **Alexandria city, Lynchburg city, Fairfax County, Virginia Beach city, Arlington County, and Henrico County** show large negative overall competitive effects. In several cases, these localities still experienced positive employment growth because favorable industry composition or statewide trends offset local underperformance.

Overall, the results demonstrate that regional employment growth cannot be interpreted from raw job change alone. Shift-share decomposition helps distinguish growth driven by broad economic conditions from growth associated with locally specific competitive performance.

## 6. Wage Structure and Change

Employment growth shows where jobs are expanding or contracting, but regional economic performance also depends on the quality and compensation of those jobs.

### Research Question

**Which Virginia industries experienced the strongest wage growth between 2019 and 2025?**

To make the comparison consistent with the employment analysis, wage growth is calculated using matched locality-industry series with published average annual pay in both 2019 and 2025.

The analysis focuses on average annual pay rather than total wages because it provides a more interpretable measure of compensation per worker.

In [40]:
# Create a matched 2019–2025 wage comparison by locality and industry.
# --------------------------------------------------------------------
# Only observations with published average annual pay in both endpoint
# years are retained so that wage growth is based on comparable records.

wage_endpoint_panel = (
    panel[
        panel["year"].isin([2019, 2025])
        & panel["avg_annual_pay"].notna()
    ]
    [
        [
            "year",
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title",
            "avg_annual_pay"
        ]
    ]
)

# Count how many endpoint years are available for each locality-industry pair.
wage_endpoint_series = (
    wage_endpoint_panel
    .groupby(
        ["area_fips", "industry_code"]
    )["year"]
    .nunique()
    .reset_index(name="endpoint_years_available")
)

# Retain only series with published pay in both 2019 and 2025.
wage_endpoint_series = wage_endpoint_series[
    wage_endpoint_series["endpoint_years_available"] == 2
]

matched_wage_panel = (
    wage_endpoint_panel
    .merge(
        wage_endpoint_series[
            ["area_fips", "industry_code"]
        ],
        on=["area_fips", "industry_code"],
        how="inner",
        validate="many_to_one"
    )
)

print(
    f"Matched locality-industry wage series: "
    f"{len(wage_endpoint_series):,}"
)

print(
    f"Matched wage endpoint observations: "
    f"{len(matched_wage_panel):,}"
)

Matched locality-industry wage series: 1,459
Matched wage endpoint observations: 2,918


In [41]:
# Reshape the matched wage panel so each locality × industry series
# has separate 2019 and 2025 average annual pay values.

locality_industry_wage_change = (
    matched_wage_panel
    .pivot(
        index=[
            "area_fips",
            "locality_name",
            "industry_code",
            "industry_title"
        ],
        columns="year",
        values="avg_annual_pay"
    )
    .reset_index()
)

# Calculate the absolute change in average annual pay.
locality_industry_wage_change["annual_pay_change"] = (
    locality_industry_wage_change[2025]
    - locality_industry_wage_change[2019]
)

# Calculate percentage wage growth between 2019 and 2025.
locality_industry_wage_change["annual_pay_growth_pct"] = (
    locality_industry_wage_change["annual_pay_change"]
    / locality_industry_wage_change[2019]
)

locality_industry_wage_change[
    [
        "locality_name",
        "industry_title",
        2019,
        2025,
        "annual_pay_change",
        "annual_pay_growth_pct"
    ]
].head(20)

year,locality_name,industry_title,2019,2025,annual_pay_change,annual_pay_growth_pct
0,Accomack County,"Agriculture, Forestry, Fishing and Hunting",47338.0,62339.0,15001.0,0.316891
1,Accomack County,Construction,40403.0,57388.0,16985.0,0.420390
2,Accomack County,Manufacturing,35794.0,53977.0,18183.0,0.507990
3,Accomack County,Wholesale Trade,44021.0,62778.0,18757.0,0.426092
4,Accomack County,Retail Trade,23040.0,32891.0,9851.0,0.427561
5,Accomack County,Information,46777.0,66047.0,19270.0,0.411955
6,Accomack County,Finance and Insurance,53297.0,79878.0,26581.0,0.498734
7,Accomack County,Real Estate and Rental and Leasing,33794.0,43275.0,9481.0,0.280553
8,Accomack County,"Professional, Scientific, and Technical Services",70838.0,87399.0,16561.0,0.233787
9,Accomack County,"Arts, Entertainment, and Recreation",18014.0,24948.0,6934.0,0.384923


### Statewide Industry Wage Benchmark

Locality-level wage changes can be volatile, particularly in smaller employment cells.

To evaluate broader industry compensation trends, average annual pay is aggregated across the matched locality sample using employment-weighted averages. This gives larger employment concentrations greater influence and produces a more representative statewide wage benchmark for each industry.

In [42]:
# Attach matched employment to the wage endpoint observations.
# -----------------------------------------------------------
# Employment weights are needed so that statewide industry pay is not
# calculated as a simple average across localities of very different sizes.

matched_wage_with_employment = (
    matched_wage_panel
    .merge(
        matched_locality_endpoint_panel[
            [
                "year",
                "area_fips",
                "industry_code",
                "annual_avg_emplvl"
            ]
        ],
        on=["year", "area_fips", "industry_code"],
        how="left",
        validate="one_to_one"
    )
)

# Calculate payroll implied by average annual pay and employment.
# This allows us to construct employment-weighted average pay by industry.
matched_wage_with_employment["implied_payroll"] = (
    matched_wage_with_employment["avg_annual_pay"]
    * matched_wage_with_employment["annual_avg_emplvl"]
)

# Aggregate employment and implied payroll by industry and year.
state_industry_wages = (
    matched_wage_with_employment
    .groupby(
        ["year", "industry_code", "industry_title"],
        as_index=False
    )
    .agg(
        matched_employment=("annual_avg_emplvl", "sum"),
        implied_payroll=("implied_payroll", "sum")
    )
)

# Calculate the employment-weighted average annual pay.
state_industry_wages["weighted_avg_annual_pay"] = (
    state_industry_wages["implied_payroll"]
    / state_industry_wages["matched_employment"]
)

state_industry_wages.head(20)

,year,industry_code,industry_title,matched_employment,implied_payroll,weighted_avg_annual_pay
0,2019,11,"Agriculture, Forestry, Fishing and Hunting",3506.0,1.270705e+08,36243.716771
1,2019,21,"Mining, Quarrying, and Oil and Gas Extraction",1112.0,7.937638e+07,71381.638489
2,2019,22,Utilities,4301.0,5.586512e+08,129888.683562
3,2019,23,Construction,165261.0,1.007612e+10,60970.969345
4,2019,31-33,Manufacturing,204146.0,1.230797e+10,60290.015910
5,2019,42,Wholesale Trade,61904.0,5.302556e+09,85657.724687
6,2019,44-45,Retail Trade,401674.0,1.237116e+10,30798.994754
7,2019,48-49,Transportation and Warehousing,107633.0,5.696192e+09,52922.356127
8,2019,51,Information,43987.0,4.930501e+09,112089.949962
9,2019,52,Finance and Insurance,119637.0,1.187111e+10,99226.058586


In [43]:
# Compare employment-weighted average annual pay by industry
# between 2019 and 2025.
# ----------------------------------------------------------
# The matched locality sample is held constant across both years,
# reducing the influence of changing disclosure coverage.

state_industry_wage_comparison = (
    state_industry_wages[
        state_industry_wages["year"].isin([2019, 2025])
    ]
    .pivot(
        index=["industry_code", "industry_title"],
        columns="year",
        values="weighted_avg_annual_pay"
    )
    .reset_index()
)

# Calculate absolute and percentage wage change.
state_industry_wage_comparison["annual_pay_change"] = (
    state_industry_wage_comparison[2025]
    - state_industry_wage_comparison[2019]
)

state_industry_wage_comparison["annual_pay_growth_pct"] = (
    state_industry_wage_comparison["annual_pay_change"]
    / state_industry_wage_comparison[2019]
)

state_industry_wage_comparison.sort_values(
    "annual_pay_growth_pct",
    ascending=False
)

year,industry_code,industry_title,2019,2025,annual_pay_change,annual_pay_growth_pct
19,99,Unclassified,42885.051343,64037.602300,21152.550956,0.493238
9,52,Finance and Insurance,99226.058586,139262.010604,40035.952019,0.403482
10,53,Real Estate and Rental and Leasing,58811.188647,82195.593821,23384.405174,0.397618
8,51,Information,112089.949962,156627.765045,44537.815083,0.397340
17,72,Accommodation and Food Services,20630.980908,28801.753858,8170.772950,0.396044
13,56,Administrative and Support and Waste Management,44961.317018,62626.226285,17664.909267,0.392891
3,23,Construction,60970.969345,83593.928767,22622.959422,0.371045
16,71,"Arts, Entertainment, and Recreation",26996.396526,36608.667850,9612.271324,0.356058
18,81,Other Services,45102.093467,60659.152986,15557.059518,0.344930
0,11,"Agriculture, Forestry, Fishing and Hunting",36243.716771,48494.146057,12250.429286,0.338001


### Answer to Research Question

**Which Virginia industries experienced the strongest wage growth between 2019 and 2025?**

Using employment-weighted average annual pay across a matched locality sample, wage growth was widespread across Virginia industries between 2019 and 2025.

Among classified industries, the strongest nominal average-pay growth occurred in:

- **Finance and Insurance:** approximately +40.3%
- **Real Estate and Rental and Leasing:** +39.8%
- **Information:** +39.7%
- **Accommodation and Food Services:** +39.6%
- **Administrative and Support and Waste Management:** +39.3%
- **Construction:** +37.1%

Several high-paying sectors also experienced substantial compensation growth. Management of Companies and Enterprises increased by approximately 32.2%, while Professional, Scientific, and Technical Services increased by approximately 27.6%.

Transportation and Warehousing presents an interesting contrast with the employment analysis. Although it was the strongest-growing industry by matched employment, its average annual pay increased by only approximately **19.0%**, one of the smaller nominal wage increases among the major sectors.

These results describe **nominal wage growth** and therefore include the effects of inflation between 2019 and 2025. Inflation-adjusted wage change would be required to determine whether purchasing power increased at the same rate.

### Inflation-Adjusted Wage Growth

Nominal wage growth does not directly measure changes in purchasing power because consumer prices also increased substantially between 2019 and 2025.

To estimate real wage growth, 2025 average annual pay is converted into **2019 dollars** using the U.S. Consumer Price Index for All Urban Consumers (CPI-U).

The CPI-U annual average increased from **255.657 in 2019 to 321.943 in 2025**, representing cumulative inflation of approximately 25.9%.

Real wage growth is therefore calculated as:

$$
g_{real}
=
\frac{1 + g_{nominal}}
{1 + \pi}
- 1
$$

where:

- $g_{nominal}$ = nominal average annual pay growth
- $\pi$ = cumulative CPI inflation between 2019 and 2025

In [44]:
# Define official BLS CPI-U annual-average index values.
# -----------------------------------------------------
# These values represent the U.S. city average for all urban consumers
# and are used to express 2025 wages in constant 2019 dollars.

CPI_2019 = 255.657
CPI_2025 = 321.943

# Calculate cumulative consumer-price inflation over the study period.
cumulative_inflation = (
    CPI_2025 / CPI_2019
) - 1

print(
    f"Cumulative CPI-U inflation, 2019–2025: "
    f"{cumulative_inflation:.1%}"
)

Cumulative CPI-U inflation, 2019–2025: 25.9%


### Real Wage Growth by Industry

Nominal wage growth is adjusted for cumulative CPI-U inflation of approximately **25.9%** between 2019 and 2025.

For each industry, 2025 average annual pay is first converted into constant 2019 dollars. The resulting real wage growth rate measures how compensation changed after accounting for the increase in consumer prices.

In [45]:
# Convert 2025 average annual pay into constant 2019 dollars.
# -----------------------------------------------------------
# Dividing by the CPI ratio removes the estimated effect of general
# consumer-price inflation from the 2025 wage level.

state_industry_wage_comparison["real_pay_2025_2019_dollars"] = (
    state_industry_wage_comparison[2025]
    * (CPI_2019 / CPI_2025)
)

# Calculate inflation-adjusted wage growth relative to 2019 pay.
state_industry_wage_comparison["real_annual_pay_growth_pct"] = (
    (
        state_industry_wage_comparison[
            "real_pay_2025_2019_dollars"
        ]
        - state_industry_wage_comparison[2019]
    )
    / state_industry_wage_comparison[2019]
)

# Display the industries from strongest to weakest real wage growth.
real_wage_growth_ranked = (
    state_industry_wage_comparison
    [
        [
            "industry_code",
            "industry_title",
            2019,
            2025,
            "annual_pay_growth_pct",
            "real_pay_2025_2019_dollars",
            "real_annual_pay_growth_pct"
        ]
    ]
    .sort_values(
        "real_annual_pay_growth_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

real_wage_growth_ranked

year,industry_code,industry_title,2019,2025,annual_pay_growth_pct,real_pay_2025_2019_dollars,real_annual_pay_growth_pct
0,99,Unclassified,42885.051343,64037.602300,0.493238,50852.670476,0.185790
1,52,Finance and Insurance,99226.058586,139262.010604,0.403482,110588.855310,0.114514
2,53,Real Estate and Rental and Leasing,58811.188647,82195.593821,0.397618,65272.047939,0.109858
3,51,Information,112089.949962,156627.765045,0.397340,124379.112229,0.109637
4,72,Accommodation and Food Services,20630.980908,28801.753858,0.396044,22871.657362,0.108607
5,56,Administrative and Support and Waste Management,44961.317018,62626.226285,0.392891,49731.887735,0.106104
6,23,Construction,60970.969345,83593.928767,0.371045,66382.474683,0.088755
7,71,"Arts, Entertainment, and Recreation",26996.396526,36608.667850,0.356058,29071.177806,0.076854
8,81,Other Services,45102.093467,60659.152986,0.344930,48169.822220,0.068017
9,11,"Agriculture, Forestry, Fishing and Hunting",36243.716771,48494.146057,0.338001,38509.512238,0.062516


### Inflation-Adjusted Wage Interpretation

After adjusting for approximately **25.9% cumulative CPI-U inflation** between 2019 and 2025, wage growth is considerably more modest than the nominal figures suggest.

Among classified industries, the strongest real average-pay gains occurred in:

- **Finance and Insurance:** approximately +11.5%
- **Real Estate and Rental and Leasing:** +11.0%
- **Information:** +11.0%
- **Accommodation and Food Services:** +10.9%
- **Administrative and Support and Waste Management:** +10.6%
- **Construction:** +8.9%

Several sectors experienced only limited real wage growth despite sizable nominal increases. Professional, Scientific, and Technical Services increased by only about **1.3% in real terms**, while Manufacturing increased by approximately **1.1%**.

Other industries did not keep pace with inflation. Real average annual pay declined in:

- **Wholesale Trade:** approximately -1.0%
- **Mining, Quarrying, and Oil and Gas Extraction:** -2.7%
- **Educational Services:** -5.5%
- **Transportation and Warehousing:** -5.5%
- **Utilities:** -6.8%

The Transportation and Warehousing result is especially notable: the sector recorded the strongest matched employment growth in Virginia, but its inflation-adjusted average pay declined over the same period. This suggests that rapid job expansion did not translate into comparable gains in worker purchasing power.

## 7. Regional Economic Analysis Summary

This notebook evaluated Virginia's regional economic performance from 2019 through 2025 using matched locality-industry comparisons designed to reduce bias from changing disclosure coverage.

Key findings include:

- **Transportation and Warehousing** recorded the strongest matched statewide employment growth, increasing by approximately **33.5%**.
- **Suffolk city, Stafford County, Frederick County, Loudoun County, and several other localities** experienced strong matched employment growth.
- **Health Care and Social Assistance** was the most common top local employment-growth driver, while **Manufacturing** was the most common dominant industry specialization.
- Location quotient analysis revealed substantial variation in local economic structure, with strong concentrations in industries such as manufacturing, agriculture, utilities, logistics, mining, and professional services.
- Shift-share analysis showed that **Loudoun County, Suffolk city, Stafford County, Prince William County, Richmond city, Chesterfield County, and Frederick County** exhibited especially strong positive regional competitive effects.
- Several localities, including **Alexandria city, Lynchburg city, Fairfax County, Virginia Beach city, Arlington County, and Henrico County**, showed negative overall competitive effects despite mixed observed employment outcomes.
- Nominal wage growth was widespread, but inflation-adjusted results were more modest. Several sectors experienced real wage gains, while Transportation and Warehousing, Utilities, Educational Services, and several other sectors did not keep pace with inflation.

Together, these results show that Virginia's regional economies differ substantially in growth, specialization, industry composition, and competitive performance. The findings provide a quantitative foundation for deeper policy interpretation and document-based economic intelligence in the next stage of the project.

In [46]:
# Save the locality × industry shift-share decomposition.
# --------------------------------------------------------
# This table preserves the observed employment change and each of the
# three shift-share components for downstream analysis and reporting.

shift_share_path = (
    OUTPUT_TABLES
    / "shift_share_locality_industry_2019_2025.csv"
)

shift_share.to_csv(
    shift_share_path,
    index=False
)

print(f"Saved shift-share results to:\n{shift_share_path}")

Saved shift-share results to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\shift_share_locality_industry_2019_2025.csv


In [47]:
# Save the 2025 locality × industry location quotient results.
# -------------------------------------------------------------
# This table preserves local employment shares, statewide benchmark
# shares, calculated location quotients, and specialization flags.

location_quotient_path = (
    OUTPUT_TABLES
    / "location_quotients_2025.csv"
)

lq_2025.to_csv(
    location_quotient_path,
    index=False
)

print(f"Saved 2025 location quotient results to:\n{location_quotient_path}")

Saved 2025 location quotient results to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\location_quotients_2025.csv


In [48]:
# Save matched locality employment growth results.
# ------------------------------------------------
# This table contains the matched 2019 and 2025 employment totals,
# absolute job change, and percentage growth for each Virginia locality.

matched_locality_growth_path = (
    OUTPUT_TABLES
    / "matched_locality_employment_growth_2019_2025.csv"
)

matched_locality_comparison.to_csv(
    matched_locality_growth_path,
    index=False
)

print(
    f"Saved matched locality employment growth results to:\n"
    f"{matched_locality_growth_path}"
)

Saved matched locality employment growth results to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\matched_locality_employment_growth_2019_2025.csv


In [49]:
# Save locality × industry employment changes.
# ---------------------------------------------
# This table includes matched endpoint employment, absolute change,
# percentage growth, and within-locality growth and decline rankings.

locality_industry_change_path = (
    OUTPUT_TABLES
    / "locality_industry_employment_change_2019_2025.csv"
)

locality_industry_change.to_csv(
    locality_industry_change_path,
    index=False
)

print(
    f"Saved locality-industry employment change results to:\n"
    f"{locality_industry_change_path}"
)

Saved locality-industry employment change results to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\locality_industry_employment_change_2019_2025.csv


In [50]:
# Save the leading growth and decline industry within each locality.
# ------------------------------------------------------------------
# These compact tables make it easy to identify the industry having
# the largest positive or negative employment contribution locally.

top_growth_driver_path = (
    OUTPUT_TABLES
    / "top_growth_driver_by_locality_2019_2025.csv"
)

top_decline_driver_path = (
    OUTPUT_TABLES
    / "top_decline_driver_by_locality_2019_2025.csv"
)

top_growth_driver_by_locality.to_csv(
    top_growth_driver_path,
    index=False
)

top_decline_driver_by_locality.to_csv(
    top_decline_driver_path,
    index=False
)

print(f"Saved top growth drivers to:\n{top_growth_driver_path}")
print(f"\nSaved top decline drivers to:\n{top_decline_driver_path}")

Saved top growth drivers to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\top_growth_driver_by_locality_2019_2025.csv

Saved top decline drivers to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\top_decline_driver_by_locality_2019_2025.csv


In [51]:
# Save aggregated locality-level regional competitive effects.
# ------------------------------------------------------------
# Positive values indicate that local industries collectively
# outperformed their statewide counterparts; negative values
# indicate relative underperformance.

locality_competitive_effects_path = (
    OUTPUT_TABLES
    / "locality_competitive_effects_2019_2025.csv"
)

locality_competitive_effects.to_csv(
    locality_competitive_effects_path,
    index=False
)

print(
    f"Saved locality competitive effects to:\n"
    f"{locality_competitive_effects_path}"
)

Saved locality competitive effects to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\locality_competitive_effects_2019_2025.csv


In [52]:
# Save nominal and real industry wage growth.
# -------------------------------------------
# The table includes 2019 and 2025 employment-weighted average annual pay,
# nominal pay growth, CPI-adjusted 2025 pay in 2019 dollars, and real growth.

industry_wage_growth_path = (
    OUTPUT_TABLES
    / "industry_wage_growth_real_and_nominal_2019_2025.csv"
)

real_wage_growth_ranked.to_csv(
    industry_wage_growth_path,
    index=False
)

print(
    f"Saved nominal and real industry wage growth results to:\n"
    f"{industry_wage_growth_path}"
)

Saved nominal and real industry wage growth results to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\industry_wage_growth_real_and_nominal_2019_2025.csv


In [53]:
# Save the strongest qualifying 2025 industry specialization
# identified for each Virginia locality.

top_specialization_path = (
    OUTPUT_TABLES
    / "top_industry_specialization_by_locality_2025.csv"
)

top_specialization_by_locality.to_csv(
    top_specialization_path,
    index=False
)

print(
    f"Saved top locality specializations to:\n"
    f"{top_specialization_path}"
)

Saved top locality specializations to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\top_industry_specialization_by_locality_2025.csv


## 8. Headline Findings for Reporting

A compact set of project-level indicators is assembled to support later visualization, documentation, and executive-summary reporting.

In [54]:
# Create a compact summary of headline quantitative findings.
# -----------------------------------------------------------
# These indicators are selected from the major analyses completed
# in Notebook 03 and will later support README and reporting content.

headline_findings = pd.DataFrame({
    "finding": [
        "Overall matched Virginia employment growth",
        "Fastest-growing matched statewide industry",
        "Fastest-growing eligible locality",
        "Largest positive locality competitive effect",
        "Most common dominant specialization",
        "Largest real wage gain among classified industries",
        "Largest real wage decline among classified industries"
    ],
    "result": [
        f"{va_growth_rate:.1%}",
        (
            matched_industry_growth_ranked
            .iloc[0]["industry_title"]
        ),
        (
            locality_growth_rank_eligible
            .sort_values("employment_growth_pct", ascending=False)
            .iloc[0]["locality_name"]
        ),
        (
            locality_competitive_effects
            .sort_values(
                "total_regional_competitive_effect",
                ascending=False
            )
            .iloc[0]["locality_name"]
        ),
        (
            top_specialization_frequency
            .iloc[0]["industry_title"]
        ),
        (
            real_wage_growth_ranked[
                real_wage_growth_ranked["industry_code"] != "99"
            ]
            .sort_values(
                "real_annual_pay_growth_pct",
                ascending=False
            )
            .iloc[0]["industry_title"]
        ),
        (
            real_wage_growth_ranked[
                real_wage_growth_ranked["industry_code"] != "99"
            ]
            .sort_values(
                "real_annual_pay_growth_pct",
                ascending=True
            )
            .iloc[0]["industry_title"]
        )
    ]
})

headline_findings

,finding,result
0,Overall matched Virginia employment growth,2.9%
1,Fastest-growing matched statewide industry,Transportation and Warehousing
2,Fastest-growing eligible locality,Suffolk city
3,Largest positive locality competitive effect,Loudoun County
4,Most common dominant specialization,Manufacturing
5,Largest real wage gain among classified indust...,Finance and Insurance
6,Largest real wage decline among classified ind...,Utilities


In [55]:
# Save the compact headline-findings table.

headline_findings_path = (
    OUTPUT_TABLES
    / "regional_economic_headline_findings.csv"
)

headline_findings.to_csv(
    headline_findings_path,
    index=False
)

print(
    f"Saved regional economic headline findings to:\n"
    f"{headline_findings_path}"
)

Saved regional economic headline findings to:
C:\Users\Chris\Documents\virginia-regional-economic-intelligence\outputs\tables\regional_economic_headline_findings.csv


In [56]:
# Verify that the major Notebook 03 output files exist.
# -----------------------------------------------------
# This final check ensures that downstream notebooks and reporting
# workflows can rely on the saved analytical products.

notebook_03_outputs = [
    matched_industry_growth_path,
    matched_locality_growth_path,
    locality_industry_change_path,
    top_growth_driver_path,
    top_decline_driver_path,
    location_quotient_path,
    shift_share_path,
    locality_competitive_effects_path,
    industry_wage_growth_path,
    top_specialization_path,
    headline_findings_path
]

output_validation = pd.DataFrame({
    "file": [
        path.name
        for path in notebook_03_outputs
    ],
    "exists": [
        path.exists()
        for path in notebook_03_outputs
    ]
})

output_validation

,file,exists
0,matched_statewide_industry_growth_2019_2025.csv,True
1,matched_locality_employment_growth_2019_2025.csv,True
2,locality_industry_employment_change_2019_2025.csv,True
3,top_growth_driver_by_locality_2019_2025.csv,True
4,top_decline_driver_by_locality_2019_2025.csv,True
5,location_quotients_2025.csv,True
6,shift_share_locality_industry_2019_2025.csv,True
7,locality_competitive_effects_2019_2025.csv,True
8,industry_wage_growth_real_and_nominal_2019_202...,True
9,top_industry_specialization_by_locality_2025.csv,True


In [57]:
# Require every major Notebook 03 output to exist before completion.

assert output_validation["exists"].all()

print("All Notebook 03 output files validated successfully.")

All Notebook 03 output files validated successfully.


## 10. Conclusion

The regional economic analysis demonstrates that Virginia's local economies followed markedly different trajectories between 2019 and 2025.

Employment growth was strongest in Transportation and Warehousing at the statewide industry level, while Health Care and Social Assistance emerged as the most geographically widespread local growth driver. At the same time, Manufacturing remained the most common source of regional industry specialization.

Matched-sample analysis showed that several localities—including Suffolk city, Stafford County, Loudoun County, Prince William County, Richmond city, Chesterfield County, and Frederick County—experienced strong employment performance. Shift-share decomposition further showed that many of these gains reflected positive local competitive effects rather than favorable industry trends alone.

The analysis also revealed important counterexamples. Some localities gained employment while still underperforming statewide industry benchmarks, demonstrating why observed job growth should not be interpreted without regional context.

Finally, wage analysis showed that nominal compensation increased broadly, but inflation-adjusted gains were much more uneven. In particular, Transportation and Warehousing combined very strong employment growth with declining real average pay, highlighting the distinction between job creation and wage-quality outcomes.

Together, the results provide a multi-dimensional view of Virginia's regional economy based on:

- employment growth,
- industry contribution,
- regional specialization,
- shift-share competitive performance,
- and real wage change.

These quantitative findings provide the foundation for the next stage of the project: integrating unstructured economic-development and policy information to help explain **why** particular regional patterns may be emerging.